# **가계신용대출 금리 트렌드**

 > </br>
 > <div style="text-align: right"> 작성날짜 : 2024년 9월 20일 </div>
 > </br>
 > <div style="text-align: right"> 작성자 : 김태영 </div>
 > </br>

## 0. 환경설정

### 0.1. 라이브러리 세팅

In [1]:
import numpy as np

import pandas as pd
from pandas import DataFrame

import os
import re
import matplotlib.pyplot as plt
from matplotlib import rc
from matplotlib import font_manager

from matplotlib import ticker
import matplotlib.ticker as mticker
from matplotlib.ticker import FuncFormatter
import matplotlib.patches as patches

import matplotlib.gridspec as gridspec

import seaborn as sns
import warnings

from adjustText import adjust_text

In [2]:
rc('font', family='Malgun Gothic')

plt.rcParams['axes.unicode_minus'] = False

In [3]:
plt.style.use('default')

In [4]:
# 특정 옵션을 원래 설정으로 되돌리기
# pd.reset_option('display.max_rows')
# pd.reset_option('display.max_columns')
# pd.reset_option('display.max_colwidth')

# 모든 옵션을 기본값으로 되돌리기
pd.reset_option('all')

C:\Users\Apro\AppData\Local\Temp\ipykernel_20900\3498853466.py:7: FutureWarning: data_manager option is deprecated and will be removed in a future version. Only the BlockManager will be available.
  pd.reset_option('all')
C:\Users\Apro\AppData\Local\Temp\ipykernel_20900\3498853466.py:7: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  pd.reset_option('all')


### 0.2. 사용자정의함수

In [5]:
def percent_to_real(df, list):
    for col in list:
        df[col] = pd.to_numeric(df[col], errors='coerce')/100
    return df

In [6]:
def erase_null_row(df):
    first_empty_index = df[df.isnull().all(axis=1)].index

    # 빈 행이 존재하면 그 인덱스부터 아래 모든 행 제거
    if not first_empty_index.empty:
        df = df[:first_empty_index[0]]

    return df

In [7]:
tot_nationwide = ['KB국민은행', '우리은행', 'SC제일은행', '신한은행', '하나은행', '한국씨티은행','NH농협은행','IBK기업은행']
tot_local = ['iM뱅크(구 대구은행)', 'BNK부산은행', '광주은행', '제주은행', '전북은행', 'BNK경남은행']
tot_internet = ['케이뱅크', '카카오뱅크', '토스뱅크']
tot_specical = ['Sh수협은행']

tot_savings = ['BNK', 'DB', 'IBK', 'JT', 'JT친애', 'KB', 'NH', 'OK', 'OSB', 'SBI'
               , '고려', '다올', '동양', '모아', '삼호', '상상인플러스', '세람', '스마트', '스타', '신한'
               , '애큐온', '예가람', '우리금융', '웰컴', '참', '청주', '키움', '키움YES', '하나', '한국투자'
               , '한성', '한화', '민국', '대원', '페퍼']

tot_card = ['경남은행', '광주은행', '롯데카드', '부산은행', '비씨카드', '삼성카드', '신한카드', '씨티은행'
            , '우리카드', '전북은행', '하나카드', '현대카드', 'IBK기업은행', 'IM뱅크', 'KB국민카드'
            , 'NH농협은행', 'SC제일은행']

tot_capital = ['롯데오토리스', '아이엠캐피탈', '한국캐피탈', '한국투자캐피탈', 'BNK캐피탈', 'KB캐피탈', '롯데캐피탈', '메리츠캐피탈'
               , '우리금융캐피탈', '하나캐피탈', '현대캐피탈', '현대커머셜', 'JB우리캐피탈', 'NH농협캐피탈'
               , '미래에셋캐피탈', 'RCI파이낸셜', '오케이캐피탈']


def categorize_company(row):
    if row['사명'] in tot_nationwide:
        return '시중은행'
    elif row['사명'] in tot_internet:
        return '인터넷은행'
    elif row['사명'] in tot_local:
        return '지방은행'
    elif row['사명'] in tot_specical:
        return '특수은행'
    else:
        return '기타'

In [8]:
def process_group_cummax(group):
    # 누적 최대값 계산
    group['cummax_mean'] = group.groupby(level=0)['mean'].cummax()

    # 누적 최대값이 역전되는 첫 번째 인덱스 찾기
    first_diff_idx = group[group['mean'] < group['cummax_mean']].first_valid_index()

    # 역전되는 시점까지의 데이터만 남기기
    if first_diff_idx is not None:
        # first_diff_idx가 튜플일 경우 위치를 찾고 슬라이싱
        first_diff_pos = group.index.get_loc(first_diff_idx)
        return group.iloc[:first_diff_pos]
    else:
        return group

In [9]:
def process_group_cummax_edit(group):
    # 누적 최대값 계산
    group['cummax_mean'] = group.groupby(level=0)['mean'].cummax()

    # 누적 최대값이 역전되는 첫 번째 인덱스 찾기
    first_diff_idx = group[group['mean'] < group['cummax_mean']].first_valid_index()
    
    # 역전되는 시점까지의 데이터만 남기기
    if first_diff_idx is not None:
        # first_diff_idx가 튜플일 경우 위치를 찾고 슬라이싱
        first_diff_pos = group.index.get_loc(first_diff_idx)
        return group.drop(first_diff_pos-1)
    else:
        return group

In [10]:
def format_as_percent_point2(value):
    return f"{value * 100:.2f}%"

def format_as_percent_point1(value):
    return f"{value * 100:.1f}%"

In [11]:
def convert_month_format(month_str):
    # 연도와 월을 분리
    year = month_str[:4]   # 앞의 4자리: '2017'
    month = month_str[4:]  # 뒤의 2자리: '03'

    # 2024년 처리
    if (year == '2024') & (month == '01'):
        return f"'{year[2:]}.{int(month)}월"  # '24.1월 형식
    elif int(year) < 2024:  # 2024년 이전은 '01', '02' 등으로 처리
        return f"{month}"
    else:
        return f"{int(month)}월"  # 그 외의 경우 '3월' 형식

## 1. 데이터 설명

### 1.1. 데이터 범위

 * 월간 신규 취급한 가계신용대출의 평균금리, 금리구간별 취급비중
 * 월간 신규 취급한 가계신용대출의 신용점수별 금리
 * 분기 중금리대출 취급실적
     - '21년 4월 금융위원회의 민간중금리 개선안인 신용점수 50%이하 차주, 업권별 금리상한 충족에 따른 중금리대출 실적 공시를 이용
     - 중금리 기준 변동 히스토리

         | 업권         | 기준('24년 3월 기준)                         | 해당 기관                                           |
         | :----------- | :------------------------------------------ | :-------------------------------------------------- |
         | 시중은행     | 가계신용대출잔액 20조원 이상, 특수은행 포함 | 국민, 신한, 우리, 하나, 농협*, IBK*              |
         | 인터넷은행   | -                                   | 토스, 카카오, K뱅크                                 |
         | 지방은행     | 가계신용대출잔액 1조원 이상                 | 경남, 광주, 대구(iM)*, 부산, 전북                        |
         | 신용카드업권 | 현금서비스, 카드론 잔액 합 5조원 이상       | 신한, 삼성, 현대, KB, 롯데                          |
         | 저축은행업권 | 가계신용대출잔액 1조원 이상                 | JT친애, SBI, OK, 다올, 애큐온, 웰컴, 페퍼, 한국투자 |
         | 캐피탈업권*   | 가계대출금 잔액 1조원 이상                  | 롯데, BNK, 농협, 우리, JB, KB, 하나, 현대           |

### 1.2. 데이터 대상
 * 가계신용대출 정의
     : 개인(가계)를 대상으로 하는 부동산, 보증기관등의 담보가 없는 순수한 신용 대출상품. 일반신용대출상품(또는 그 유사한) 및 한도대출로 정의

 * 가계신용대출 취급 업권 분류
     - 은행업권(시중은행, 인터넷은행, 지방은행)
     - 여전사업권(신용카드업권, 캐피탈업권)
     - 저축은행업권
     - 대부업권(데이터 부재로 본 분석에서 다루지 않음)
     - 일반적으로 저축은행업권, 캐피탈업권, 대부업권을 SP업권으로 간주


 * 업권별 주요사 선정
     - 주요사 선정 기준: 업권별 가계신용대출(또는 그 유사) 일정 잔고 이상
     - 시중은행 5곳, 인터넷은행 3곳, 지방은행 5곳, 신용카드사 5곳, 저축은행 8곳, 캐피탈 8곳 총 34개사

         | 업권         | 기준('24년 3월말 기준)                         | 해당 기관                                           |
         | :----------- | :------------------------------------------ | :-------------------------------------------------- |
         | 시중은행     | 가계신용대출잔고 20조원 이상, 특수은행 포함 | 국민, 신한, 우리, 하나, 농협*             |
         | 인터넷은행   | -                                   | 토스, 카카오, K뱅크                                 |
         | 지방은행     | 가계신용대출잔고 1조원 이상                 | 경남, 광주, 대구(iM)*, 부산, 전북                        |
         | 신용카드업권 | 현금서비스, 카드론 잔고 합 5조원 이상       | 신한, 삼성, 현대, KB, 롯데                          |
         | 저축은행업권 | 가계신용대출잔고 1조원 이상                 | JT친애, OK, SBI, 다올, 애큐온, 웰컴, 페퍼, 한국투자 |
         | 캐피탈업권*   | 가계대출금 잔고 1조원 이상                  | 롯데, BNK, 농협, 우리, JB, KB, 하나, 현대           |

           * 농협은 특수은행으로 분류되나, 가계신용대출의 규모나 행태가 시중은행과 유사하므로 시중은행으로 분류
           * 대구은행(현 iM뱅크)의 경우 시중은행으로 전환되었으나, 지방은행의 특성을 그대로 띠고 있어 지방은행으로 분류
           * 캐피탈업권은 자료미비로 가계대출금으로 대체

### 1.3. 테이블
 * 개별 테이블

     | 변수(테이블)          | 업 권       | 내 용                                          | 주 기 | 비 고                                                            |
     | :----------------- | :-------: | :--------------------------------------------- | :---------: | :-------------------------------------------------------------- |
     | bank_cs_standard       | 은행업권    | 기관별 일반신용대출 신용점수별 금리                     | 월         | 시중은행, 인터넷은행, 지방은행 구분                    |
     | bank_cs_minus      | 은행업권    | 기관별 한도대출 신용점수별 금리                        | 월         | 시중은행, 인터넷은행, 지방은행 구분                                     |
     | bank_int_standard      | 은행업권    | 기관별 일반신용대출 금리구간별 취급비중                    | 월         | 시중은행, 인터넷은행, 지방은행 구분                                    |
     | bank_int_minus     | 은행업권    | 기관별 한도대출 금리구간별 취급비중                       | 월         | 시중은행, 인터넷은행, 지방은행 구분                                     |
     | bank_bal           | 은행업권    | 기관별 가계신용대출 잔고                              | 분기       | 시중은행, 인터넷은행, 지방은행 구분                                      |
     | bank_med_cs        | 은행업권    | 기관별 가계신용대출 중금리 신용점수별 금리                          | 분기       | 시중은행, 인터넷은행, 지방은행 구분                                      |
     | bank_med_outp      | 은행업권    | 기관별 가계신용대출 중금리 취급실적                             | 분기       | 시중은행, 인터넷은행, 지방은행 구분                                      |
     | card_cs_loan       | 신용카드업권 | 기관별 카드론(장기카드대출) 신용점수별 금리                     | 월         |                                                                 |
     | card_cs_cash       | 신용카드업권 | 기관별 현금서비스(단기카드대출) 신용점수별 금리                 | 월         |                                                                 |
     | card_int_loan      | 신용카드업권 | 기관별 카드론(장기카드대출) 금리구간별 취급비중(이용인원기준)     | 월         | 금액 비중이 아닌 이용인원 비중                                   |
     | card_int_cash      | 신용카드업권 | 기관별 현금서비스(단기카드대출) 금리구간별 취급비중(이용인원기준) | 월         | 금액 비중이 아닌 이용인원 비중                                   |
     | card_bal           | 신용카드업권 | 기관별 주요카드자산(현금서비스, 카드론) 잔고                              | 분기       |                                         |
     | card_med_cs        | 신용카드업권 | 기관별 중금리대출 신용점수별 금리                                  | 분기       |                                                                 |
     | card_med_outp      | 신용카드업권 | 기관별 중금리대출 취급실적                                      | 분기       |                                                                 |
     | card_outp          | 신용카드업권 | 기관별 주요카드자산(현금서비스, 카드론) 취급실적                          | 분기       | 기관 분기보고서 참고                                                |
     | savings_cs         | 저축은행업권 | 기관별 일반신용대출 신용점수별 금리                             | 월         |                                                                 |
     | savings_int        | 저축은행업권 | 기관별 금리구간별 취급비중                                        | 월         |                                                                 |
     | savings_bal        | 저축은행업권 | 기관별 잔고(가계신용대출)                                       | 분기       |                                                                 |
     | savings_med_cs     | 저축은행업권 | 기관별 중금리대출 신용점수별 금리                                   | 분기       |                                                                 |
     | savings_med_outp   | 저축은행업권 | 기관별 중금리대출 취급실적                                      | 분기       |                                                                 |
     | capital_cs         | 캐피탈업권   | 기관별 일반신용대출 신용점수별 금리                             | 월         |                                                                 |
     | capital_int        | 캐피탈업권   | 기관별 금리구간별 취급비중                                    | 월       |                                                                    |
     | capital_bal        | 캐피탈업권   | 기관별 잔고(가계대출금)                                         | 분기       | 가계대출금은 개인(가계)를 대상으로 하는 모든 유형의 대출을 의미 |
     | capital_med_cs     | 캐피탈업권   | 기관별 중금리대출 신용점수별 금리                                  | 분기       |                                                                 |
     | capital_med_outp   | 캐피탈업권   | 기관별 중금리대출 취급실적                                  | 분기       |                                                                 |

###
 * 통합 테이블 및 참고 테이블

     | 변수(테이블) | 업 권                                 | 내 용                                         | 주 기 | 비 고                         |
     | :----------- | :-----------------------------------: | :------------------------------------------- | :---------: | :--------------------------- |
     | tot_cs       | 전 업권                               | 기관별 신용점수별 금리                                 | 월         |                              |
     | tot_int      | 은행업권, 신용카드업권, 저축은행업권        | 기관별 금리구간별 취급비중                                | 월         | 신용카드업권은 이용인원 비중 |
     | tot_med_cs   | 전 업권                               | 기관별 중금리대출 신용점수별 금리                             | 분기       |                              |
     | tot_med_outp   | 전 업권                               | 기관별 중금리대출 취급실적                              | 분기       |                              |
     | tot_bal      | 전 업권                               | 기관별 가계신용대출 잔고, 주요사 선정 및 가계신용대출 시장 트렌드 추적에 이용 | 분기       | 캐피탈업권의 경우 가계대출금 |
     | med_dics  | 전 업권                               | 업권별 금리, 신용점수 중금리 인정 상한선                             | 반기       | 금융위원회 고시                             |

        * tot 테이블명에 _major를 붙이면 주요사만 추출


###
 * 테이블별 다운로드 경로
     | 테이블 | 다운로드 경로                                 | 비 고                         |
     | :----------- | :----------------------------------- | :--------------------------- |
     | 저축은행업권(금리구간별 취급비중) | 저축은행중앙회_상품공시_대출_가계신용대출_금리대별 취급비중 / 월 |   |
     | 저축은행업권(신용점수별 금리현황) | 저축은행중앙회_상품공시_대출_가계신용대출_저축은행별금리현황 / 월 |   |
     | 저축은행업권(실적) | ?? |   |
     | 저축은행업권(잔액) | 예금보험공사_금융회사 종합정보_저축은행_경영정보_주요통계 / 분기 |   |
     | 저축은행업권(중금리 취급실적) | 저축은행중앙회 / 분기 |   |
     | 신용카드업권(금리구간별 취급비중_카드론) | 여신금융협회_신용카드상품 비교공시_신용카드회원 분포현황_적용금리대별 회원분포현황_카드론  | 프린트창에서 html다운로드 후 csv변환  |
     | 신용카드업권(금리구간별 취급비중_현금서비스) | 여신금융협회_신용카드상품 비교공시_신용카드회원 분포현황_적용금리대별 회원분포현황_현금서비스 | 프린트창에서 html다운로드 후 csv변환 |
     | 신용카드업권(신용점수별 금리현황_카드론) | 여신금융협회_신용카드상품 비교공시_카드대출리볼빙 신용점수별 금리_카드론 |  |
     | 신용카드업권(신용점수별 금리현황_현금서비스) | 여신금융협회_신용카드상품 비교공시_카드대출리볼빙 신용점수별 금리_현금서비스 |  |
     | 신용카드업권(실적) | DART_카드사별 검색 | pdf보고 수치 수작업 |
     | 신용카드업권(잔액) | 금감원 금융통계정보시스템_비은행_신용카드사_부문별 자산현황_카드자산_카드론+현금서비스 |
     | 신용카드업권(중금리 취급실적) | 여신금융협회_신용대출상품 비교공시_중금리 신용대출 현황 | 프린트창에서 html다운로드 후 csv변환 / 공시되는 분기안에 반드시 다운로드 |
     | 신용카드업권(회원등급-참고자료) | 여신금융협회_신용카드상품 비교공시_신용카드회원 분포현황_회원등급별 분포현황 / ?주기가 따로 없음
     | 은행업권(금리구간별 취급비중_마이너스) | 은행연합회_금리/수수료 비교공시_대출금리비교_금리구간별 취급비중_가계대출금리(공시기준:신규취급액 기준, 대출종류:신용한도대출) |
     | 은행업권(금리구간별 취급비중_일반신용) | 은행연합회_금리/수수료 비교공시_대출금리비교_금리구간별 취급비중_가계대출금리(공시기준:신규취급액 기준, 대출종류:일반신용대출) |
     | 은행업권(신용점수별 취급비중_마이너스) | 은행연합회_금리/수수료 비교공시_대출금리비교_은행별 비교공시_가계대출금리(공시기준:신규취급액 기준, 대출종류:신용한도대출, 상세구분:대출금리) |
     | 은행업권(신용점수별 취급비중_일반신용) | 은행연합회_금리/수수료 비교공시_대출금리비교_은행별 비교공시_가계대출금리(공시기준:신규취급액 기준, 대출종류:일반신용대출, 상세구분:대출금리) |
     | 은행업권(잔액) | 금감원 금융통계정보시스템_은행_국내은행_재무현황_유형별 대출채권_여신종별 원화대출금(가계자금_신용대출 등) |
     | 은행업권(중금리 취급실적) | 은행연합회_금리/수수료 비교공시_금리구간별 취급비중_대출금리비교_맞춤상품검색_중금리대출 |
     | 캐피탈업권(금리구간별 취급비중 - 참고자료) | 여신금융협회_신용대출상품 비교공시_적용금리대별 회원분포현황 | 프린트창에서 html다운로드 후 csv변환
     | 캐피탈업권(신용점수별 금리현황) | 여신금융협회_신용대출상품 비교공시_신용평가사 신용점수 기준 |
     | 캐피탈업권(잔액) | 금감원 금융통계정보시스템_비은행_리스사/할부금융사(두 부문 각각 다운로드)_재무현황_부문별 자산현황_대출채권(대여금_가계대출금) | xls를 csv로 변환 |

 * 딕셔너리 정리
     - table_list : 개별테이블명(key), 내용(value)
     - industry_list : 업권별 테이블(은행업권, 신용카드업권, 저축은행업권, 캐피탈업권)
     - tot_list : cs, int, bal, med_cs, med_outp

In [ ]:
table_list = {
  'bank_cs_standard': '은행업권 - 기관별 일반신용대출 신용점수별 금리',
  'bank_cs_minus': '은행업권 - 기관별 한도대출 신용점수별 금리',
  'bank_int_standard': '은행업권 - 기관별 일반신용대출 금리구간별 취급비중',
  'bank_int_minus': '은행업권 - 기관별 한도대출 금리구간별 취급비중',
  'bank_bal': '은행업권 - 기관별 가계신용대출 잔액',
  'bank_med_cs': '은행업권 - 기관별 가계신용대출 중금리 신용점수별 금리',
  'bank_med_outp': '은행업권 - 기관별 가계신용대출 중금리 취급실적',
  'card_cs_loan': '신용카드업권 - 기관별 카드론(장기카드대출) 신용점수별 금리',
  'card_cs_cash': '신용카드업권 - 기관별 현금서비스(단기카드대출) 신용점수별 금리',
  'card_int_loan': '신용카드업권 - 기관별 카드론(장기카드대출) 금리구간별 취급비중(이용인원기준)',
  'card_int_cash': '신용카드업권 - 기관별 현금서비스(단기카드대출) 금리구간별 취급비중(이용인원기준)',
  'card_bal': '신용카드업권 - 기관별 주요카드자산(현금서비스, 카드론) 잔액',
  'card_med_cs': '신용카드업권 - 기관별 중금리대출 신용점수별 금리',
  'card_med_outp': '신용카드업권 - 기관별 중금리대출 취급실적',
  'card_outp': '신용카드업권 - 기관별 주요카드자산(현금서비스, 카드론) 취급실적',
  'savings_cs': '저축은행업권 - 기관별 일반신용대출 신용점수별 금리',
  'savings_int': '저축은행업권 - 기관별 금리구간별 취급비중',
  'savings_bal': '저축은행업권 - 기관별 잔액(가계신용대출)',
  'savings_med_cs': '저축은행업권 - 기관별 중금리대출 신용점수별 금리',
  'savings_med_outp': '저축은행업권 - 기관별 중금리대출 취급실적',
  'capital_cs': '캐피탈업권 - 기관별 일반신용대출 신용점수별 금리',
  'capital_bal': '캐피탈업권 - 기관별 잔액(가계대출금)',
  'capital_med_cs': '캐피탈업권 - 기관별 중금리대출 신용점수별 금리',
  'capital_med_outp': '캐피탈업권 - 기관별 중금리대출 취급실적'
}

In [ ]:
industry_list = {
  '은행업권': ['bank_cs_standard', 'bank_cs_minus', 'bank_int_standard', 'bank_int_minus', 'bank_bal', 'bank_med_cs', 'bank_med_outp'],
  '신용카드업권': ['card_cs_loan', 'card_cs_cash', 'card_int_loan', 'card_int_cash', 'card_bal', 'card_med_cs', 'card_med_outp', 'card_outp'],
  '저축은행업권': ['savings_cs', 'savings_int', 'savings_bal', 'savings_med_cs', 'savings_med_outp'],
  '캐피탈업권': ['capital_cs', 'capital_bal', 'capital_med_cs', 'capital_med_outp']
}

In [ ]:
tot_list = {
  'cs': ['bank_cs_standard', 'bank_cs_minus', 'card_cs_loan', 'card_cs_cash', 'savings_cs', 'capital_cs'],
  'int': ['bank_int_standard', 'bank_int_minus', 'card_int_loan', 'card_int_cash', 'savings_int'],
  'bal': ['bank_bal', 'card_bal', 'savings_bal', 'capital_bal'],
  'med_cs': ['bank_med_cs', 'card_med_cs', 'savings_med_cs', 'capital_med_cs'],
  'med_outp': ['bank_med_outp', 'card_med_outp', 'savings_med_outp', 'capital_med_outp']
}

### 1.4. 데이터 출처

 * 금융위원회
 * 금융감독원
 * 예금보험공사
 * 은행연합회
 * 여신금융협회
 * 저축은행중앙회
 * 개별기관 분기보고서(DART)

## 2. 개별 테이블 생성

### 2.0. 설정

#### 2.0.1. 수집날짜 및 파일경로

In [ ]:
#########################################
collect_date = '20250204'
#########################################

file_path_bank_cs_standard = f"./RAW/RAW_{collect_date}/은행업권/은행업권(신용점수별 금리현황_일반신용)"
file_path_bank_cs_minus = f"./RAW/RAW_{collect_date}/은행업권/은행업권(신용점수별 금리현황_마이너스)"
file_path_bank_int_standard = f"./RAW/RAW_{collect_date}/은행업권/은행업권(금리구간별 취급비중_일반신용)"
file_path_bank_int_minus = f"./RAW/RAW_{collect_date}/은행업권/은행업권(금리구간별 취급비중_마이너스)"
file_path_bank_bal = f"./RAW/RAW_{collect_date}/은행업권/은행업권(잔액)"
file_path_bank_med = f"./RAW/RAW_{collect_date}/은행업권/은행업권(중금리 취급실적)"

file_path_card_cs_loan = f"./RAW/RAW_{collect_date}/신용카드업권/신용카드업권(신용점수별 금리현황_카드론)"
file_path_card_cs_cash = f"./RAW/RAW_{collect_date}/신용카드업권/신용카드업권(신용점수별 금리현황_현금서비스)"
file_path_card_int_loan = f"./RAW/RAW_{collect_date}/신용카드업권/신용카드업권(금리구간별 취급비중_카드론)"
file_path_card_int_cash = f"./RAW/RAW_{collect_date}/신용카드업권/신용카드업권(금리구간별 취급비중_현금서비스)"
file_path_card_bal = f"./RAW/RAW_{collect_date}/신용카드업권/신용카드업권(잔액)"
file_path_card_med = f"./RAW/RAW_{collect_date}/신용카드업권/신용카드업권(중금리 취급실적)"

file_path_savings_cs = f"./RAW/RAW_{collect_date}/저축은행업권/저축은행업권(신용점수별 금리현황)"
file_path_savings_int = f"./RAW/RAW_{collect_date}/저축은행업권/저축은행업권(금리구간별 취급비중)"
file_path_savings_bal = f"./RAW/RAW_{collect_date}/저축은행업권/저축은행업권(잔액)"
file_path_savings_med = f"./RAW/RAW_{collect_date}/저축은행업권/저축은행업권(중금리 취급실적)"

file_path_capital_cs = f"./RAW/RAW_{collect_date}/캐피탈업권/캐피탈업권(신용점수별 금리현황)"
file_path_capital_int = f"./RAW/RAW_{collect_date}/캐피탈업권/캐피탈업권(금리구간별 취급비중 - 참고자료)"
file_path_capital_bal = f"./RAW/RAW_{collect_date}/캐피탈업권/캐피탈업권(잔액)"

#### 2.0.2. 사명 리스트 및 표준화

In [12]:
tot_nationwide = ['KB국민은행', '우리은행', 'SC제일은행', '신한은행', '하나은행', '한국씨티은행','NH농협은행','IBK기업은행']
tot_local = ['iM뱅크(구 대구은행)', 'BNK부산은행', '광주은행', '제주은행', '전북은행', 'BNK경남은행']
tot_internet = ['케이뱅크', '카카오뱅크', '토스뱅크']
tot_specical = ['Sh수협은행']

tot_savings = ['BNK', 'DB', 'IBK', 'JT', 'JT친애', 'KB', 'NH', 'OK', 'OSB', 'SBI'
               , '고려', '다올', '동양', '모아', '삼호', '상상인플러스', '세람', '스마트', '스타', '신한'
               , '애큐온', '예가람', '우리금융', '웰컴', '참', '청주', '키움', '키움YES', '하나', '한국투자'
               , '한성', '한화', '민국', '대원', '페퍼']

tot_card = ['경남은행', '광주은행', '롯데카드', '부산은행', '비씨카드', '삼성카드', '신한카드', '씨티은행'
            , '우리카드', '전북은행', '하나카드', '현대카드', 'IBK기업은행', 'IM뱅크', 'KB국민카드'
            , 'NH농협은행', 'SC제일은행']

tot_capital = ['롯데오토리스', '아이엠캐피탈', '한국캐피탈', '한국투자캐피탈', 'BNK캐피탈', 'KB캐피탈', '롯데캐피탈', '메리츠캐피탈'
               , '우리금융캐피탈', '하나캐피탈', '현대캐피탈', '현대커머셜', 'JB우리캐피탈', 'NH농협캐피탈'
               , '미래에셋캐피탈', 'RCI파이낸셜', '오케이캐피탈']

### 2.1. 은행업권(시중은행, 인터넷은행, 지방은행)

#### 2.1.1. 신용점수별 금리_일반신용대출

In [ ]:
file_list_bank_cs_standard = [file for file in os.listdir(file_path_bank_cs_standard) if file.endswith(('.csv', '.xlsx', '.xls'))]

bank_cs_standard_list_1 = []
bank_cs_standard_list_2 = []

for file in file_list_bank_cs_standard:
    date_match = re.search(r'\d{6}', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass
    
    if pd.to_numeric(date) <= 202206:
        df = pd.read_excel(f'{file_path_bank_cs_standard}/{file}', skiprows=1)
        df.drop(['Unnamed: 1', '서민금융제외평균금리', 'Unnamed: 9'], axis=1, inplace=True)
        df.rename(columns={'Unnamed: 0':'사명'}, inplace=True)

        df['연월'] = date
        df['사명'] = df['사명'].str.strip()
        df = df[~(df['사명']=='KDB산업은행')]
        df['업권'] = df.apply(categorize_company, axis=1)
        df['상품구분'] = '일반신용'
        df['CB기관'] = 'NULL'
        df['평균신용점수'] = np.nan

        percent_to_real(df, ['1~2등급', '3~4등급', '5~6등급', '7~8등급 ', '9~10등급', '평균금리'])

        bank_cs_standard_list_1.append(df)
        
    elif pd.to_numeric(date) >= 202207:
        df = pd.read_excel(f'{file_path_bank_cs_standard}/{file}', skiprows=1)
        df.drop(['Unnamed: 1', '서민금융제외평균금리', 'Unnamed: 15'], axis=1, inplace=True)
        df.rename(columns={'Unnamed: 0':'사명', 'Unnamed: 13':'평균신용점수', 'Unnamed: 14':'CB기관'}, inplace=True)

        df['연월'] = date
        df['사명'] = df['사명'].str.strip()
        df = df[~(df['사명']=='KDB산업은행')]
        df['업권'] = df.apply(categorize_company, axis=1)
        df['상품구분'] = '일반신용'
        if '700점 이하 평균금리' in df.columns:
            df.drop(columns='700점 이하 평균금리', inplace=True)
        else:
            pass

        percent_to_real(df, ['1000~951점', '950~901점', '900~851점', '850~801점'
                             , '800~751점', '750~701점', '700~651점', '650~601점', '600점이하', '평균금리'])

        bank_cs_standard_list_2.append(df)

bank_cs_standard_1 = pd.concat(bank_cs_standard_list_1, ignore_index=True)
bank_cs_standard_1 = bank_cs_standard_1.melt(id_vars = ['사명', '업권', '상품구분', '연월', 'CB기관', '평균금리', '평균신용점수']
                                 , value_vars = ['1~2등급', '3~4등급', '5~6등급', '7~8등급 ', '9~10등급']
                                 , var_name = '신용점수구간', value_name = '구간평균금리')
        
        
bank_cs_standard_2 = pd.concat(bank_cs_standard_list_2, ignore_index=True)
bank_cs_standard_2.loc[bank_cs_standard_2['CB기관'].str.contains('KCB'), 'CB기관'] = 'KCB'
bank_cs_standard_2.loc[bank_cs_standard_2['CB기관'].str.contains('NICE'), 'CB기관'] = 'NICE'
bank_cs_standard_2.loc[bank_cs_standard_2['CB기관'].str.contains('나이스'), 'CB기관'] = 'NICE'
bank_cs_standard_2 = bank_cs_standard_2[bank_cs_standard_2['CB기관'] != 'KIS']

bank_cs_standard_2 = bank_cs_standard_2.melt(id_vars = ['사명', '업권', '상품구분', '연월', 'CB기관', '평균금리', '평균신용점수']
                                 , value_vars = ['1000~951점', '950~901점', '900~851점', '850~801점', '800~751점', '750~701점', '700~651점', '650~601점', '600점이하']
                                 , var_name = '신용점수구간', value_name = '구간평균금리')

bank_cs_standard = pd.concat([bank_cs_standard_1, bank_cs_standard_2], ignore_index=True)

bank_cs_standard

#### 2.1.2. 신용점수별 금리_한도대출

In [ ]:
file_list_bank_cs_minus = [file for file in os.listdir(file_path_bank_cs_minus) if file.endswith(('.csv', '.xlsx', '.xls'))]

bank_cs_minus_list = []

for file in file_list_bank_cs_minus:
    date_match = re.search(r'\d{6}', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass

    df = pd.read_excel(f'{file_path_bank_cs_minus}/{file}', skiprows=1, engine='openpyxl')
    df.drop(['Unnamed: 1', 'Unnamed: 14'], axis=1, inplace=True)
    df.rename(columns={'Unnamed: 0':'사명', '카드사명':'사명', 'Unnamed: 12':'평균신용점수', 'Unnamed: 13':'CB기관'}, inplace=True)

    df['연월'] = date
    df['사명'] = df['사명'].str.strip()
    df = df[~(df['사명']=='KDB산업은행')]
    df['업권'] = df.apply(categorize_company, axis=1)
    df['상품구분'] = '한도'

    if '700점 이하 평균금리' in df.columns:
        df.drop(columns='700점 이하 평균금리', inplace=True)
    else:
        pass

    percent_to_real(df, ['1000~951점', '950~901점', '900~851점', '850~801점'
                         , '800~751점', '750~701점', '700~651점', '650~601점', '600점이하', '평균금리'])

    bank_cs_minus_list.append(df)

bank_cs_minus = pd.concat(bank_cs_minus_list, ignore_index=True)
bank_cs_minus.loc[bank_cs_minus['CB기관'].str.contains('KCB'), 'CB기관'] = 'KCB'
bank_cs_minus.loc[bank_cs_minus['CB기관'].str.contains('NICE'), 'CB기관'] = 'NICE'
bank_cs_minus.loc[bank_cs_minus['CB기관'].str.contains('나이스'), 'CB기관'] = 'NICE'
bank_cs_minus = bank_cs_minus[bank_cs_minus['CB기관'] != 'KIS']

bank_cs_minus = bank_cs_minus.melt(id_vars = ['사명', '업권', '상품구분', '연월', 'CB기관', '평균금리', '평균신용점수']
                                   , value_vars = ['1000~951점', '950~901점', '900~851점', '850~801점', '800~751점', '750~701점', '700~651점', '650~601점', '600점이하']
                                   , var_name = '신용점수구간', value_name = '구간평균금리')

bank_cs_minus

#### 2.1.3. 금리구간별 취급비중_일반신용대출

In [ ]:
file_list_bank_int_standard = [file for file in os.listdir(file_path_bank_int_standard) if file.endswith(('.csv', '.xlsx', '.xls'))]

bank_int_standard_list = []

for file in file_list_bank_int_standard:
    date_match = re.search(r'\d{6}', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass

    df = pd.read_excel(f'{file_path_bank_int_standard}/{file}')
    df.drop(['서민금융제외 평균금리', '합계'], axis=1, inplace=True)
    df.rename(columns={'은행':'사명'}, inplace=True)

    df['연월'] = date
    df['상품구분'] = '일반신용'
    df['사명'] = df['사명'].str.strip()
    df = df[~(df['사명']=='KDB산업은행')]
    df['업권'] = df.apply(categorize_company, axis=1)

    percent_to_real(df, ['4%미만', '4~5%미만', '5~6%미만', '6~7%미만'
                         , '7~8%미만', '8~9%미만', '9~10%미만', '10%이상', '평균금리'])

    bank_int_standard_list.append(df)

bank_int_standard = pd.concat(bank_int_standard_list, ignore_index=True)

bank_int_standard = bank_int_standard.melt(id_vars = ['사명', '업권', '상품구분', '연월']
                                   , value_vars = ['4%미만', '4~5%미만', '5~6%미만', '6~7%미만', '7~8%미만', '8~9%미만', '9~10%미만', '10%이상']
                                   , var_name = '금리구간', value_name = '취급비중')

bank_int_standard

#### 2.1.4. 금리구간별 취급비중_한도대출

In [ ]:
file_list_bank_int_minus = [file for file in os.listdir(file_path_bank_int_minus) if file.endswith(('.csv', '.xlsx', '.xls'))]

bank_int_minus_list = []

for file in file_list_bank_int_minus:
    date_match = re.search(r'\d{6}', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass

    df = pd.read_excel(f'{file_path_bank_int_minus}/{file}')
    df.drop(['합계'], axis=1, inplace=True)
    df.rename(columns={'은행':'사명'}, inplace=True)

    df['연월'] = date
    df['사명'] = df['사명'].str.strip()
    df = df[~(df['사명']=='KDB산업은행')]
    df['업권'] = df.apply(categorize_company, axis=1)
    df['상품구분'] = '한도'

    percent_to_real(df, ['4%미만', '4~5%미만', '5~6%미만', '6~7%미만'
                         , '7~8%미만', '8~9%미만', '9~10%미만', '10%이상', '평균금리'])

    bank_int_minus_list.append(df)

bank_int_minus = pd.concat(bank_int_minus_list, ignore_index=True)

bank_int_minus = bank_int_minus.melt(id_vars = ['사명', '업권', '상품구분', '연월']
                                   , value_vars = ['4%미만', '4~5%미만', '5~6%미만', '6~7%미만', '7~8%미만', '8~9%미만', '9~10%미만', '10%이상']
                                   , var_name = '금리구간', value_name = '취급비중')

# bank_int_minus = bank_int_minus.dropna(subset=['취급비중'])

bank_int_minus

#### 2.1.5. 가계신용대출 잔액

In [ ]:
file_list_bank_bal = [file for file in os.listdir(file_path_bank_bal) if file.endswith(('.csv'))] # xls파일은 취급하지 않음

bank_bal = pd.read_csv(f'{file_path_bank_bal}/{file_list_bank_bal[0]}', encoding='euc-kr', skiprows=[0,25,26])
bank_bal = bank_bal[['금융회사명','2023년03월말','2023년06월말','2023년09월말','2023년12월말','2024년03월말','2024년06월말']]
bank_bal.columns = ['사명', 202303, 202306, 202309, 202312, 202403,202406]
bank_bal[[202303, 202306, 202309, 202312, 202403, 202406]] = bank_bal[[202303, 202306, 202309, 202312, 202403, 202406]].apply(pd.to_numeric, errors='coerce')
bank_bal[[202303, 202306, 202309, 202312, 202403, 202406]] = bank_bal[[202303, 202306, 202309, 202312, 202403, 202406]] * 1000000

bank_bal['사명'] = bank_bal['사명'].replace({'경남은행':'BNK경남은행','부산은행':'BNK부산은행','중소기업은행':'IBK기업은행','농협은행주식회사':'NH농협은행',
                                            '국민은행':'KB국민은행','한국스탠다드차타드은행':'SC제일은행','수협은행':'Sh수협은행','대구은행(19990301~20240604)':'iM뱅크(구 대구은행)','아이엠뱅크':'iM뱅크(구 대구은행)',
                                            '주식회사 카카오뱅크':'카카오뱅크','주식회사 케이뱅크':'케이뱅크','토스뱅크 주식회사':'토스뱅크'})

bank_bal['업권'] = bank_bal.apply(categorize_company, axis=1)

bank_bal = bank_bal.melt(id_vars = ['사명', '업권'], value_vars = [202303,  202306, 202309, 202312, 202403, 202406], var_name = '연월', value_name = '잔액')

bank_bal = bank_bal.dropna(subset=['사명', '잔액'])

bank_bal

#### 2.1.6. 중금리

In [ ]:
file_list_bank_med = [file for file in os.listdir(file_path_bank_med) if file.endswith(('.csv', '.xlsx', '.xls'))]


df = pd.read_excel(f'{file_path_bank_med}/{file_list_bank_med[0]}', skiprows=2)
df = df.iloc[:,0:20]

df.drop(['Unnamed: 5','Unnamed: 7','Unnamed: 9','Unnamed: 11','Unnamed: 13','Unnamed: 15','Unnamed: 17','Unnamed: 19'], axis=1, inplace=True)
df.columns = ['사명','상품명','취급금액','취급건수'
              ,'900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하']
df = df[df['상품명']=='민간중금리']
df.drop(columns='상품명', axis=1, inplace=True)
df['업권'] = df.apply(categorize_company, axis=1)
df['연월'] = '2024Q2'
df['사명'] = df['사명'].str.strip()
df['취급금액'] = pd.to_numeric(df['취급금액']) * 1000000
df = df[['사명','업권','연월','취급금액','취급건수'
         ,'900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하']]

bank_med_outp = df.iloc[:,0:5]
bank_med_cs = df.drop(['취급금액','취급건수'], axis=1)
bank_med_cs = bank_med_cs.melt(id_vars = ['사명', '업권', '연월']
                                , value_vars = ['900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하']
                                , var_name = '신용점수구간', value_name = '구간평균금리')


bank_med_cs['구간평균금리'] = pd.to_numeric(bank_med_cs['구간평균금리']).replace(0, np.nan)

bank_med_outp
bank_med_cs

### 2.2. 신용카드업권

#### 2.2.1. 신용점수별 금리_카드론

In [ ]:
file_list_card_cs_loan = [file for file in os.listdir(file_path_card_cs_loan) if file.endswith(('.csv', '.xlsx', '.xls'))]

card_cs_loan_list = []

for file in file_list_card_cs_loan:
    date_match = re.search(r'\d{6}', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass

    df = pd.read_excel(f'{file_path_card_cs_loan}/{file}', skiprows=2)

    df['연월'] = date
    df['업권'] = '신용카드'
    df['상품구분'] = '카드론'
    df['사명'] = df['사명'].str.strip()

    df.rename(columns={'신용정보회사':'CB기관', '카드사명':'사명'}, inplace=True)

    if '700점 이하 평균금리' in df.columns:
        df.drop(columns='700점 이하 평균금리', inplace=True)
    else:
        pass

    percent_to_real(df, ['900점 초과', '801점~900점', '701점~800점', '601점~700점'
                         , '501점~600점', '401점~500점', '301점~400점', '300점 이하', '평균금리'])

    df = df.melt(id_vars = ['사명', '업권', '상품구분', '연월', 'CB기관','평균금리']
                            , value_vars = ['900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하']
                            , var_name = '신용점수구간', value_name = '구간평균금리')

    df['구간평균금리'] = pd.to_numeric(df['구간평균금리']).replace(0, np.nan)

    card_cs_loan_list.append(df)

card_cs_loan = pd.concat(card_cs_loan_list, ignore_index=True)
card_cs_loan.loc[card_cs_loan['CB기관'].str.contains('KCB'), 'CB기관'] = 'KCB'
card_cs_loan.loc[card_cs_loan['CB기관'].str.contains('NICE'), 'CB기관'] = 'NICE'
card_cs_loan.loc[card_cs_loan['CB기관'].str.contains('나이스'), 'CB기관'] = 'NICE'
card_cs_loan = card_cs_loan[card_cs_loan['CB기관'] != 'KIS']

card_cs_loan

#### 2.2.2. 신용점수별 금리_현금서비스

In [ ]:
file_list_card_cs_cash = [file for file in os.listdir(file_path_card_cs_cash) if file.endswith(('.csv', '.xlsx', '.xls'))]

card_cs_cash_list = []

for file in file_list_card_cs_cash:
    date_match = re.search(r'\d{6}', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass

    df = pd.read_excel(f'{file_path_card_cs_cash}/{file}', skiprows=2)

    df['연월'] = date
    df['업권'] = '신용카드'
    df['상품구분'] = '현금서비스'
    df['사명'] = df['사명'].str.strip()

    df.rename(columns={'신용정보회사':'CB기관', '카드사명':'사명'}, inplace=True)

    if '700점 이하 평균금리' in df.columns:
        df.drop(columns='700점 이하 평균금리', inplace=True)
    else:
        pass

    percent_to_real(df, ['900점 초과', '801점~900점', '701점~800점', '601점~700점'
                         , '501점~600점', '401점~500점', '301점~400점', '300점 이하', '평균금리'])

    df = df.melt(id_vars = ['사명', '업권', '상품구분', '연월', 'CB기관','평균금리']
                        , value_vars = ['900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하']
                        , var_name = '신용점수구간', value_name = '구간평균금리')

    df['구간평균금리'] = pd.to_numeric(df['구간평균금리']).replace(0, np.nan)

    card_cs_cash_list.append(df)

card_cs_cash = pd.concat(card_cs_cash_list, ignore_index=True)
card_cs_cash.loc[card_cs_cash['CB기관'].str.contains('KCB'), 'CB기관'] = 'KCB'
card_cs_cash.loc[card_cs_cash['CB기관'].str.contains('NICE'), 'CB기관'] = 'NICE'
card_cs_cash.loc[card_cs_cash['CB기관'].str.contains('나이스'), 'CB기관'] = 'NICE'
card_cs_cash = card_cs_cash[card_cs_cash['CB기관'] != 'KIS']

card_cs_cash

#### 2.2.3. 금리대별 취급비중_카드론

In [ ]:
file_list_card_int_loan = [file for file in os.listdir(file_path_card_int_loan) if file.endswith(('.csv', '.xlsx', '.xls'))]

card_int_loan_list = []

for file in file_list_card_int_loan:
    date_match = re.search(r'\d{6}', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass

    df = pd.read_csv(f'{file_path_card_int_loan}/{file}', encoding='euc-kr')

    df.columns = ['사명','이용회원','기간구분','10%미만','10~12%','12~14%','14~16%','16~18%','18~20%']
    df.drop(columns=['이용회원'], axis=1, inplace=True)
    df['사명'] = df['사명'].ffill()
    df['사명'] = df['사명'].str.strip()
    df = df[df['기간구분']=='전체']

    percent_to_real(df, ['10%미만','10~12%','12~14%','14~16%','16~18%','18~20%'])
    df['업권'] = '신용카드'
    df['상품구분'] = '카드론'
    df['연월'] = date
    df.drop(columns='기간구분', inplace=True)
    df = df.melt(id_vars = ['사명', '업권', '상품구분', '연월']
                , value_vars = ['10%미만','10~12%','12~14%','14~16%','16~18%','18~20%']
                , var_name = '금리구간', value_name = '취급비중')

    card_int_loan_list.append(df)
    
card_int_loan = pd.concat(card_int_loan_list, ignore_index=True)    
    
card_int_loan

#### 2.2.4. 금리대별 취급비중_현금서비스

In [ ]:
file_list_card_int_cash = [file for file in os.listdir(file_path_card_int_cash) if file.endswith(('.csv', '.xlsx', '.xls'))]

card_int_cash_list = []

for file in file_list_card_int_cash:
    date_match = re.search(r'\d{6}', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass

    df = pd.read_csv(f'{file_path_card_int_cash}/{file}', encoding='euc-kr')

    df.columns = ['사명','회원구분','10%미만','10~12%','12~14%','14~16%','16~18%','18~20%']
    df['사명'] = df['사명'].ffill()
    df['사명'] = df['사명'].str.strip()
    df = df[df['회원구분']=='이용회원']

    percent_to_real(df, ['10%미만','10~12%','12~14%','14~16%','16~18%','18~20%'])
    df['업권'] = '신용카드'
    df['상품구분'] = '현금서비스'
    df['연월'] = date
    df = df.melt(id_vars = ['사명', '업권', '상품구분', '연월']
                , value_vars = ['10%미만','10~12%','12~14%','14~16%','16~18%','18~20%']
                , var_name = '금리구간', value_name = '취급비중')

    card_int_cash_list.append(df)
    
card_int_cash = pd.concat(card_int_cash_list, ignore_index=True)    
    
card_int_cash

#### 2.2.5. 주요카드자산(카드론, 현금서비스) 잔액

In [ ]:
file_list_card_bal = [file for file in os.listdir(file_path_card_bal) if file.endswith(('.csv'))] # xls파일은 취급하지 않음

card_bal = pd.read_csv(f'{file_path_card_bal}/{file_list_card_bal[0]}', encoding='euc-kr', skiprows=[0]) # 추후 파일 누적시 for loop 구성 필요
card_bal = card_bal[['금융회사명', '구분', '2023년03월말', '2023년06월말', '2023년09월말', '2023년12월말', '2024년03월말', '2024년06월말']]
card_bal.columns = ['사명', '상품구분', 202303, 202306, 202309, 202312, 202403, 202406]
card_bal[[202303, 202306, 202309, 202312, 202403, 202406]] = card_bal[[202303, 202306, 202309, 202312, 202403, 202406]] * 1000000

card_bal['사명'] = card_bal['사명'].str.strip()
card_bal['사명'] = card_bal['사명'].str.replace("㈜", "", regex=False)
card_bal['사명'] = card_bal['사명'].str.replace("케이비국민카드", "KB국민카드", regex=False)

card_bal['업권'] = '신용카드'
card_bal['업권'] = card_bal['업권'].str.replace(" ", "")

card_bal = card_bal.melt(id_vars = ['사명','업권','상품구분'], value_vars = [202303,202306,202309,202312,202403,202406], var_name = '연월', value_name = '잔액')

card_bal_tot = card_bal.groupby(['사명','업권','연월'], as_index=False).sum()
card_bal_tot['상품구분'] = '가계신용대출'
card_bal_tot['잔액'] = pd.to_numeric(card_bal_tot['잔액'])

card_bal = pd.concat([card_bal, card_bal_tot], axis=0, ignore_index=True)
card_bal

#### 2.2.6. 중금리 취급실적 (캐피탈 함께 출력)

In [ ]:
file_list_card_med = [file for file in os.listdir(file_path_card_med) if file.endswith(('.csv'))]

df = pd.read_csv(f'{file_path_card_med}/{file_list_card_med[0]}', encoding='euc-kr', skiprows=[0]) # 추후 파일 누적시 for loop 구성 필요

df.columns = ['사명','취급금액','취급건수','구분'
              ,'900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하','상세보기']
df = df[df['구분']=='평균금리']
df.drop(columns='구분', axis=1, inplace=True)
df['업권'] = '신용카드'
df['연월'] = '2024Q2'
df['사명'] = df['사명'].str.strip()
df['취급금액'] = pd.to_numeric(df['취급금액']) * 1000000
df = df[['사명','업권','연월','취급금액','취급건수'
         ,'900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하']]
to_num_list = ['900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하']
df[to_num_list] = df[to_num_list].apply(pd.to_numeric, errors='coerce')
percent_to_real(df, ['900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하'])
card_med_outp = df[df['사명'].str.contains('카드')].iloc[:,0:5]
card_med_cs = df[df['사명'].str.contains('카드')].drop(['취급금액','취급건수'], axis=1)
card_med_cs = card_med_cs.melt(id_vars = ['사명', '업권', '연월']
                                , value_vars = ['900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하']
                                , var_name = '신용점수구간', value_name = '구간평균금리')

capital_med_outp = df[~df['사명'].str.contains('카드')].iloc[:,0:5]

capital_med_cs = df[~df['사명'].str.contains('카드')].drop(['취급금액','취급건수'], axis=1)
capital_med_cs = capital_med_cs.melt(id_vars = ['사명', '업권', '연월']
                                , value_vars = ['900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하']
                                , var_name = '신용점수구간', value_name = '구간평균금리')

capital_med_outp

#### 2.2.7. 주요카드자산(카드론, 현금서비스) 취급실적

In [ ]:
card_outp_dic = {
    "사명" : ["롯데카드", "롯데카드", "롯데카드", "롯데카드"
              , "삼성카드", "삼성카드", "삼성카드", "삼성카드"
              , "신한카드", "신한카드", "신한카드", "신한카드"
              , "KB국민카드", "KB국민카드", "KB국민카드", "KB국민카드"
              , "현대카드", "현대카드", "현대카드", "현대카드"],
    "연월" : [202403, 202403, 202406, 202406
            , 202403, 202403, 202406, 202406
            , 202403, 202403, 202406, 202406
            , 202403, 202403, 202406, 202406
            , 202403, 202403, 202406, 202406],
    "상품구분" : ["현금서비스", "카드론", "현금서비스", "카드론"
                , "현금서비스", "카드론", "현금서비스", "카드론"
                , "현금서비스", "카드론", "현금서비스", "카드론"
                , "현금서비스", "카드론", "현금서비스", "카드론"
                , "현금서비스", "카드론", "현금서비스", "카드론"],
    "실적" : [1509900000000, 1512500000000, 1570500000000, 1652200000000
            , 1820323000000, 2427443000000, 2001126000000, 1976882000000
            , 3219500000000, 2147800000000, 3189400000000, 2272600000000
            , 2528110000000, 1706262000000, 2436769000000, 1762415000000
            , 1559200000000, 1675600000000, 1602400000000, 1772800000000]
}

card_outp = pd.DataFrame(card_outp_dic)
card_outp

### 2.3. 저축은행업권

#### 2.3.1. 신용점수별 금리

In [ ]:
file_list_savings_cs = [file for file in os.listdir(file_path_savings_cs) if file.endswith(('.csv', '.xlsx', '.xls'))]

savings_cs_list = []

for file in file_list_savings_cs:
    date_match = re.search(r'\d{6}', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass

    if pd.to_numeric(date) > 202012:
        df = pd.read_html(f'{file_path_savings_cs}/{file}', encoding='utf-8')
        df = df[0]
        df.columns = df.columns.droplevel(level=0)
        df.columns = ['사명', '구분', '900점 초과', '801점~900점', '701점~800점', '601점~700점'
                              , '501점~600점', '401점~500점', '301점~400점', '300점 이하', '금리', 'CB기관']
        df = df[df['구분'].isin(['평균대출금리', '기준금리(a)', '가산금리(b)'])]
        df = df.pivot(index=['사명'], columns='구분', values='금리').reset_index()
        df['연월'] = date
        
        percent_to_real(df, ['평균대출금리', '기준금리(a)', '가산금리(b)'])
        # 데이터프레임 리스트에 추가
        savings_cs_list.append(df)
        
    elif pd.to_numeric(date) < 202101:
        pass

# 모든 데이터프레임을 하나로 병함

savings_cs = pd.concat(savings_cs_list, ignore_index=True)

savings_cs.to_csv('저축은행 기준금리 변화.csv', encoding='euc-kr', index=False)

In [ ]:
savings_cs['사명'].unique()

In [ ]:
savings_cs = savings_cs[(savings_cs['사명']=='SBI') & (savings_cs['연월']>'202301')]

y_list = ['평균대출금리', '기준금리(a)', '가산금리(b)']
x_col = '연월'

plt.figure(figsize=(20,6))
for y in y_list:
    plt.plot(savings_cs[x_col], savings_cs[y], marker='o', linestyle='-', label=y)
    
plt.legend()
plt.grid(True)

plt.show()



In [ ]:
file_list_savings_cs = [file for file in os.listdir(file_path_savings_cs) if file.endswith(('.csv', '.xlsx', '.xls'))]

savings_cs_list_1 = []
savings_cs_list_2 = []

for file in file_list_savings_cs:
    date_match = re.search(r'\d{6}', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass

    if pd.to_numeric(date) <= 202012:
        df = pd.read_html(f'{file_path_savings_cs}/{file}', encoding='utf-8')
        df = df[0]
        df.columns = df.columns.droplevel(level=0)
        df.columns = ['사명', '구분', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '평균금리']
        df = df[df['구분'].isin(['평균대출금리', '기준금리(a)', '가산금리(b)'])]

        percent_to_real(df, ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '평균금리'])

        df['업권'] = '저축은행'
        df['상품구분'] = '일반신용'
        df['CB기관'] = 'NULL'

        # 날짜 열 추가
        df['연월'] = date

        # 데이터프레임 리스트에 추가
        savings_cs_list_1.append(df)
        
    elif pd.to_numeric(date) >= 202101:
        df = pd.read_html(f'{file_path_savings_cs}/{file}', encoding='utf-8')
        df = df[0]
        df.columns = df.columns.droplevel(level=0)
        df.columns = ['사명', '구분', '900점 초과', '801점~900점', '701점~800점', '601점~700점'
                      , '501점~600점', '401점~500점', '301점~400점', '300점 이하', '평균금리', 'CB기관']
        df = df[df['구분'].isin(['평균대출금리', '기준금리(a)', '가산금리(b)'])]

        percent_to_real(df, ['900점 초과', '801점~900점', '701점~800점', '601점~700점'
                             , '501점~600점', '401점~500점', '301점~400점', '300점 이하', '평균금리'])

        df['업권'] = '저축은행'
        df['상품구분'] = '일반신용'

        # 날짜 열 추가
        df['연월'] = date

        # 데이터프레임 리스트에 추가
        savings_cs_list_2.append(df)

# 모든 데이터프레임을 하나로 병함

savings_cs_1 = pd.concat(savings_cs_list_1, ignore_index=True)
savings_cs_1 = savings_cs_1[savings_cs_1['구분']=='평균대출금리']
savings_cs_1.drop(['구분'], axis=1, inplace=True)
savings_cs_1 = savings_cs_1.melt(id_vars = ['사명', '업권', '상품구분', '연월','CB기관','평균금리']
                                , value_vars = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']
                                , var_name = '신용점수구간', value_name = '구간평균금리')

savings_cs_2 = pd.concat(savings_cs_list_2, ignore_index=True)
savings_cs_2 = savings_cs_2[savings_cs_2['구분']=='평균대출금리']
savings_cs_2.drop(['구분'], axis=1, inplace=True)
savings_cs_2.loc[savings_cs_2['CB기관'].notna() & savings_cs_2['CB기관'].str.contains('KCB'), 'CB기관'] = 'KCB'
savings_cs_2.loc[savings_cs_2['CB기관'].notna() & savings_cs_2['CB기관'].str.contains('NICE'), 'CB기관'] = 'NICE'
savings_cs_2.loc[savings_cs_2['CB기관'].notna() & savings_cs_2['CB기관'].str.contains('나이스'), 'CB기관'] = 'NICE'
savings_cs_2 = savings_cs_2[savings_cs_2['CB기관'] != 'KIS']
savings_cs_2 = savings_cs_2.melt(id_vars = ['사명', '업권', '상품구분', '연월','CB기관','평균금리']
                                , value_vars = ['900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하']
                                , var_name = '신용점수구간', value_name = '구간평균금리')

savings_cs = pd.concat([savings_cs_1, savings_cs_2], ignore_index=True)

savings_cs

#### 2.3.2. 금리대별 취급비중

In [ ]:
file_list_savings_int = [file for file in os.listdir(file_path_savings_int) if file.endswith(('.csv', '.xlsx', '.xls'))]

savings_int_list = []

for file in file_list_savings_int:
    date_match = re.search(r'\d{6}', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass

    df = pd.read_html(f'{file_path_savings_int}/{file}', encoding='utf-8')
    df = df[0]
    df.rename({'10% 이하.1':'14% 이하'}, inplace=True)
    df.columns = ['사명', '10%이하', '12%이하', '14%이하', '16%이하', '18%이하', '20%이하', '계']
    df = df[['사명', '10%이하', '12%이하', '14%이하', '16%이하', '18%이하', '20%이하']]

    percent_to_real(df, ['10%이하', '12%이하', '14%이하', '16%이하', '18%이하', '20%이하'])

    df['업권'] = '저축은행'
    df['상품구분'] = '일반신용'

    df['연월'] = date

    df = df.melt(id_vars = ['사명', '업권', '상품구분', '연월']
                                , value_vars = ['10%이하', '12%이하', '14%이하', '16%이하', '18%이하', '20%이하']
                                , var_name = '금리구간', value_name = '취급비중')

    # 데이터프레임 리스트에 추가
    savings_int_list.append(df)

# 모든 데이터프레임을 하나로 병함
savings_int = pd.concat(savings_int_list, ignore_index=True)
savings_int

#### 2.3.3. 가계신용대출 잔액

In [ ]:
file_list_savings_bal = [file for file in os.listdir(file_path_savings_bal) if file.endswith(('.csv', '.xlsx', '.xls'))]

savings_bal_list = []
for file in file_list_savings_bal:
    date_match = re.search(r'\d{4}Q\d', file) # 분기표현식
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass

    df = pd.read_excel(f'{file_path_savings_bal}/{file}', skiprows=3)

    df = df.iloc[:, [0,5]]

    df.columns = ['사명', '잔액']
    df['잔액'] = pd.to_numeric(df['잔액']) * 100000000
    df['사명'] = df['사명'].replace({'키움예스':'키움YES'})
    df['업권'] = '저축은행'
    df['연월'] = date
    # quarter_to_month = {'Q1':'03', 'Q2':'06', 'Q3':'09', 'Q4':'12'}
    df['연월'] = df['연월'].str.replace('Q1','03', regex=True) \
                               .replace('Q2','06', regex=True) \
                               .replace('Q3','09', regex=True) \
                               .replace('Q4','12', regex=True)
    df = df[['사명','업권','연월','잔액']]

    savings_bal_list.append(df)

savings_bal = pd.concat(savings_bal_list, ignore_index=True)

savings_bal

#### 2.3.4. 중금리 취급실적

In [ ]:
file_list_savings_med = [file for file in os.listdir(file_path_savings_med) if file.endswith(('.csv', '.xlsx', '.xls'))]

savings_med_outp_list = []
savings_med_cs_list = []

for file in file_list_savings_med:
    date_match = re.search(r'\d{4}Q\d', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass

    df = pd.read_html(f'{file_path_savings_med}/{file}', encoding='utf-8')
    df = df[0]

    df.columns = df.columns.droplevel(0)

    df[['사명', '상품구분']] = df['저축은행'].str.extract(r'\[(.*?)\]\s*(.*)')
    df.drop('저축은행', axis=1, inplace=True)
    df['업권'] = '저축은행'
    df['연월'] = date
    df = df.rename(columns={'금액(백만원)':'취급금액', '건수':'취급건수', 'CB 기관':'CB기관', '저축은행.1':'금리종류', '900점초과':'900점 초과'
                            , '801점 ~ 900점':'801점~900점', '701점 ~ 800점':'701점~800점', '601점 ~ 700점':'601점~700점'
                            , '501점 ~ 600점':'501점~600점', '401점 ~ 500점':'401점~500점', '301점 ~ 400점':'301점~400점','300점이하':'300점 이하'})
    df = df[['사명','상품구분','금리종류','업권','연월','취급금액','취급건수','900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하','CB기관']]
    df = df[(df['상품구분']=='민간중금리')&(df['금리종류']=='평균금리')]
    df.drop(columns=['상품구분','금리종류'], axis=1, inplace=True)
    df['취급금액'] = pd.to_numeric(df['취급금액']) * 1000000

    df1 = df.iloc[:,0:5]

    df2 = df.melt(id_vars = ['사명', '업권', '연월','CB기관']
                    , value_vars = ['900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하']
                    , var_name = '신용점수구간', value_name = '구간평균금리')

    savings_med_outp_list.append(df1)
    savings_med_cs_list.append(df2)
    
# 모든 데이터프레임을 하나로 병함
savings_med_outp = pd.concat(savings_med_outp_list, ignore_index=True)
savings_med_cs = pd.concat(savings_med_cs_list, ignore_index=True)
savings_med_outp

### 2.4. 캐피탈업권

#### 2.4.1. 신용점수별 금리

In [ ]:
file_list_capital_cs = [file for file in os.listdir(file_path_capital_cs) if file.endswith(('.csv', '.xlsx', '.xls'))]

capital_cs_list = []

for file in file_list_capital_cs:
    date_match = re.search(r'\d{6}', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass

    df = pd.read_excel(f'{file_path_capital_cs}/{file}', skiprows=2)

    df['연월'] = date
    df['업권'] = '캐피탈'
    df['상품구분'] = '일반신용'

    df.rename(columns={'신용정보회사':'CB기관', '기관명':'사명'}, inplace=True)

    if '700점 이하 평균금리' in df.columns:
        df.drop(columns='700점 이하 평균금리', inplace=True)
    else:
        pass

    percent_to_real(df, ['900점 초과', '801점~900점', '701점~800점', '601점~700점'
                         , '501점~600점', '401점~500점', '301점~400점', '300점 이하', '평균금리'])

    capital_cs_list.append(df)

capital_cs = pd.concat(capital_cs_list, ignore_index=True)
capital_cs.loc[capital_cs['CB기관'].str.contains('KCB'), 'CB기관'] = 'KCB'
capital_cs.loc[capital_cs['CB기관'].str.contains('NICE'), 'CB기관'] = 'NICE'
capital_cs.loc[capital_cs['CB기관'].str.contains('나이스'), 'CB기관'] = 'NICE'
capital_cs = capital_cs[capital_cs['CB기관'] != 'KIS']

capital_cs = capital_cs[~capital_cs['사명'].str.contains('카드')]

capital_cs = capital_cs.melt(id_vars = ['사명', '업권', '상품구분', '연월','CB기관','평균금리']
                                , value_vars = ['900점 초과', '801점~900점', '701점~800점', '601점~700점', '501점~600점', '401점~500점', '301점~400점', '300점 이하']
                                , var_name = '신용점수구간', value_name = '구간평균금리')

capital_cs['구간평균금리'] = capital_cs['구간평균금리'].replace(0, np.nan)

capital_cs

#### 2.4.2. 금리대별 취급비중

In [ ]:
file_list_capital_int = [file for file in os.listdir(file_path_capital_int) if file.endswith(('.csv', '.xlsx', '.xls'))]

capital_int_list = []

for file in file_list_capital_int:
    date_match = re.search(r'\d{6}', file)
    if date_match:
        date_str = date_match.group()
        date = date_str
    else:
        pass
    
    df = pd.read_csv(f'{file_path_capital_int}/{file}',encoding='euc-kr') # 추후 파일 누적시 for loop 구성 필요
    df.columns = ['사명','8%미만','8~12%','12~16%','16~20%','계']
    df = df[~df['사명'].str.contains('카드')]
    df.drop(columns=['계'], axis=1, inplace=True)

    percent_to_real(df, ['8%미만','8~12%','12~16%','16~20%'])
    df['업권'] = '캐피탈'
    df['상품구분'] = '일반신용'
    df['연월'] = date
    df = df.melt(id_vars = ['사명', '업권', '상품구분', '연월']
                , value_vars = ['8%미만','8~12%','12~16%','16~20%']
                , var_name = '금리구간', value_name = '취급비중')

    capital_int_list.append(df)

capital_int = pd.concat(capital_int_list, ignore_index=True)
    
capital_int

#### 2.4.3. 가계대출금 잔액

In [ ]:
file_list_capital_bal = [file for file in os.listdir(file_path_capital_bal) if file.endswith(('.csv', '.xlsx', '.xls'))]

capital_bal_list = []
for file in file_list_capital_bal:
    df = pd.read_csv(f'{file_path_capital_bal}/{file}', skiprows=1, encoding='euc-kr')
    capital_bal_list.append(df)

capital_bal = pd.concat(capital_bal_list, axis=0)
capital_bal.reset_index(drop=True)

capital_bal = capital_bal[['금융회사명', '2023년03월말', '2023년06월말', '2023년09월말', '2023년12월말', '2024년03월말', '2024년06월말']]
capital_bal['업권'] = '캐피탈'

capital_bal.columns = ['사명', 202303, 202306, 202309, 202312, 202403, 202406, '업권']
capital_bal['사명'] = capital_bal['사명'].str.replace("㈜", "", regex=False)

name_standard = {'디지비캐피탈':'아이엠캐피탈','비엔케이캐피탈':'BNK캐피탈','케이비캐피탈':'KB캐피탈','제이비우리캐피탈':'JB우리캐피탈','엔에이치농협캐피탈':'NH농협캐피탈','알씨아이파이낸셜서비스코리아':'RCI파이낸셜'}
capital_bal['사명'] = capital_bal['사명'].replace(name_standard)

capital_bal = capital_bal.melt(id_vars = ['사명','업권'], value_vars = [202303,202306,202309,202312,202403,202406], var_name = '연월', value_name = '잔액')

capital_bal['잔액'] = pd.to_numeric(capital_bal['잔액'], errors='coerce') * 1000000

capital_bal

#### 2.4.3. 중금리 취급실적

 - 2.2.6. 신용카드업권 처리시 같이 처리

## 3. 통합 테이블

 - 개별테이블에서는 원값 보존 차원에서 신용점수구간 및 금리구간을 코드화 하지 않았으나, 통합 테이블부터는 시각화의 편리성, 데이터 신뢰성 제고를 위해 코드화 진행

### 3.1. 신용점수별 금리 통합 테이블

In [13]:
# tot_cs = pd.concat([bank_cs_standard.drop(columns='평균신용점수')
#                     , bank_cs_minus.drop(columns='평균신용점수')
#                     , card_cs_loan
#                     , card_cs_cash
#                     , savings_cs
#                     , capital_cs
#                     ], axis=0)
tot_cs

cs_code = {'1000~951점':'011'
           , '950~901점':'012'
           , '900~851점':'021'
           , '850~801점':'022'
           , '800~751점':'031'
           , '750~701점':'032'
           , '700~651점':'041'
           , '650~601점':'042'
           , '600점이하':'050' # 은행업권의 600점이하는 501~600점으로 간주하여 통합
           , '900점 초과':'010'
           , '801점~900점':'020'
           , '701점~800점':'030'
           , '601점~700점':'040'
           , '501점~600점':'050'
           , '401점~500점':'060'
           , '301점~400점':'070'
           , '300점 이하':'080'
}
cs_code_rev = {v: k for k, v in cs_code.items()}
cs_code_easy ={'011':'9'
               , '012':'9'
               , '010':'9'
               , '021':'8'
               , '022':'8'
               , '020':'8'
               , '031':'7'
               , '032':'7'
               , '030':'7'
               , '041':'6'
               , '042':'6'
               , '040':'6'
               , '050':'5'
               , '060':'4'
               , '070':'3'
               , '080':'3L'
    }
cs_code_easy_rev = {'1':'900점대'
                    ,'2':'800점대'
                    ,'3':'700점대'
                    ,'4':'600점대'
                    ,'5':'500점대'
                    ,'6':'400점대'
                    ,'7':'300점대'
                    ,'8':'300점이하'
}
tot_cs['신용점수구간'] = tot_cs['신용점수구간'].replace(cs_code)
tot_cs

NameError: name 'bank_cs_standard' is not defined

### 3.2. 금리대별 취급비중 통합 테이블

In [ ]:
tot_int = pd.concat([bank_int_standard
                    , bank_int_minus
                    , card_int_loan
                    , card_int_cash
                    , savings_int
                    ], axis=0)
int_code = {'4%미만':'010'
            , '4~5%미만':'012'
            , '5~6%미만':'013'
            , '6~7%미만':'014'
            , '7~8%미만':'015'
            , '8~9%미만':'016'
            , '9~10%미만':'017'
            , '10%이상':'018'
            , '10%미만':'017'
            , '10~12%':'018'
            , '12~14%':'019'
            , '14~16%':'020'
            , '16~18%':'021'
            , '18~20%':'022'
            , '10%이하':'017'
            , '12%이하':'018'
            , '14%이하':'019'
            , '16%이하':'020'
            , '18%이하':'021'
            , '20%이하':'022'
}
int_code_ul = {'010':{'Lower Bound':0,'Upper Bound':0.04}
               , '012':{'Lower Bound':0.04,'Upper Bound':0.05}
               , '013':{'Lower Bound':0.05,'Upper Bound':0.06}
               , '014':{'Lower Bound':0.06,'Upper Bound':0.07}
               , '015':{'Lower Bound':0.07,'Upper Bound':0.08}
               , '016':{'Lower Bound':0.08,'Upper Bound':0.09}
               , '017':{'Lower Bound':0.09,'Upper Bound':0.10}
               , '018':{'Lower Bound':0.10,'Upper Bound':0.12}
               , '019':{'Lower Bound':0.12,'Upper Bound':0.14}
               , '020':{'Lower Bound':0.14,'Upper Bound':0.16}
               , '021':{'Lower Bound':0.16,'Upper Bound':0.18}
               , '022':{'Lower Bound':0.18,'Upper Bound':0.20}
}
tot_int['금리구간'] = tot_int['금리구간'].replace(int_code)
tot_int

### 3.3. 중금리 취급실적 통합 테이블

In [ ]:
tot_med_cs = pd.concat([bank_med_cs
                        , card_med_cs
                        , savings_med_cs.drop(columns='CB기관', axis=1)
                        , capital_med_cs
                        ], axis=0)
tot_med_cs['신용점수구간'] = tot_med_cs['신용점수구간'].replace(cs_code)
tot_med_cs

In [ ]:
tot_med_outp = pd.concat([bank_med_outp
                          , card_med_outp
                          , savings_med_outp
                          , capital_med_outp
                         ], axis=0)
tot_med_outp

### 3.4. 가계신용대출 잔액 통합 테이블

In [ ]:
tot_bal = pd.concat([bank_bal
                    , card_bal[card_bal['상품구분']=='가계신용대출'].drop(columns='상품구분')
                    , savings_bal
                    , capital_bal
                    ], axis=0)
tot_bal

### 3.5. 주요 기관 추출 통합테이블

In [ ]:
major_nationwide = tot_bal[(tot_bal['업권']=='시중은행') & (tot_bal['연월']==202403) &(tot_bal['잔액']>20000000000000)]['사명'].unique().tolist()
major_local = tot_bal[(tot_bal['업권']=='지방은행') & (tot_bal['연월']==202403) &(tot_bal['잔액']>1000000000000)]['사명'].unique().tolist()
major_internet = tot_bal[(tot_bal['업권']=='인터넷은행') & (tot_bal['연월']==202403)]['사명'].unique().tolist()
major_savings = tot_bal[(tot_bal['업권']=='저축은행') & (tot_bal['연월']=='202403') &(tot_bal['잔액']>1000000000000)]['사명'].unique().tolist()
major_card = tot_bal[(tot_bal['업권']=='신용카드') & (tot_bal['연월']==202403) &(tot_bal['잔액']>5000000000000)]['사명'].unique().tolist()
major_capital = tot_bal[(tot_bal['업권']=='캐피탈') & (tot_bal['연월']==202403) &(tot_bal['잔액']>1000000000000)]['사명'].unique().tolist()

In [ ]:
tot_cs_major = pd.concat([ tot_cs[(tot_cs['업권']=='시중은행') & (tot_cs['사명'].isin(major_nationwide))]
                          , tot_cs[(tot_cs['업권']=='인터넷은행') & (tot_cs['사명'].isin(major_internet))]
                          , tot_cs[(tot_cs['업권']=='지방은행') & (tot_cs['사명'].isin(major_local))]
                          , tot_cs[(tot_cs['업권']=='신용카드') & (tot_cs['사명'].isin(major_card))]
                          , tot_cs[(tot_cs['업권']=='캐피탈') & (tot_cs['사명'].isin(major_capital))]
                          , tot_cs[(tot_cs['업권']=='저축은행') & (tot_cs['사명'].isin(major_savings))] ], axis=0)
tot_int_major = pd.concat([ tot_int[(tot_int['업권']=='시중은행') & (tot_int['사명'].isin(major_nationwide))]
                          , tot_int[(tot_int['업권']=='인터넷은행') & (tot_int['사명'].isin(major_internet))]
                          , tot_int[(tot_int['업권']=='지방은행') & (tot_int['사명'].isin(major_local))]
                          , tot_int[(tot_int['업권']=='신용카드') & (tot_int['사명'].isin(major_card))]
                          , tot_int[(tot_int['업권']=='캐피탈') & (tot_int['사명'].isin(major_capital))]
                          , tot_int[(tot_int['업권']=='저축은행') & (tot_int['사명'].isin(major_savings))] ], axis=0)
tot_med_cs_major = pd.concat([ tot_med_cs[(tot_med_cs['업권']=='시중은행') & (tot_med_cs['사명'].isin(major_nationwide))]
                          , tot_med_cs[(tot_med_cs['업권']=='인터넷은행') & (tot_med_cs['사명'].isin(major_internet))]
                          , tot_med_cs[(tot_med_cs['업권']=='지방은행') & (tot_med_cs['사명'].isin(major_local))]
                          , tot_med_cs[(tot_med_cs['업권']=='신용카드') & (tot_med_cs['사명'].isin(major_card))]
                          , tot_med_cs[(tot_med_cs['업권']=='캐피탈') & (tot_med_cs['사명'].isin(major_capital))]
                          , tot_med_cs[(tot_med_cs['업권']=='저축은행') & (tot_med_cs['사명'].isin(major_savings))] ], axis=0)
tot_med_outp_major = pd.concat([ tot_med_outp[(tot_med_outp['업권']=='시중은행') & (tot_med_outp['사명'].isin(major_nationwide))]
                          , tot_med_outp[(tot_med_outp['업권']=='인터넷은행') & (tot_med_outp['사명'].isin(major_internet))]
                          , tot_med_outp[(tot_med_outp['업권']=='지방은행') & (tot_med_outp['사명'].isin(major_local))]
                          , tot_med_outp[(tot_med_outp['업권']=='신용카드') & (tot_med_outp['사명'].isin(major_card))]
                          , tot_med_outp[(tot_med_outp['업권']=='캐피탈') & (tot_med_outp['사명'].isin(major_capital))]
                          , tot_med_outp[(tot_med_outp['업권']=='저축은행') & (tot_med_outp['사명'].isin(major_savings))] ], axis=0)
tot_bal_major = pd.concat([ tot_bal[(tot_bal['업권']=='시중은행') & (tot_bal['사명'].isin(major_nationwide))]
                          , tot_bal[(tot_bal['업권']=='인터넷은행') & (tot_bal['사명'].isin(major_internet))]
                          , tot_bal[(tot_bal['업권']=='지방은행') & (tot_bal['사명'].isin(major_local))]
                          , tot_bal[(tot_bal['업권']=='신용카드') & (tot_bal['사명'].isin(major_card))]
                          , tot_bal[(tot_bal['업권']=='캐피탈') & (tot_bal['사명'].isin(major_capital))]
                          , tot_bal[(tot_bal['업권']=='저축은행') & (tot_bal['사명'].isin(major_savings))] ], axis=0)

In [ ]:
tot_cs_major[tot_cs_major['업권']=='저축은행']

## 4. 시장 트렌드

### 4.1. 신규취급 평균금리 트렌드

#### 4.1.1. 그래프

In [ ]:
class int_step_graph:
    def __init__(self, ax, df, mode, mode_spec, mode_spec_color, mode_standard, standard_list, industry_list, start_month, industry_color, indiv_color, industry_width, indiv_width, standard_color, standard_width, ylim_auto, fontsize, numlabels_pos, xlabels_pos, numlabels_on=True):
        self.ax = ax
        self.df = df
        self.mode = mode # 'open'(개별기업 펼침), 'close'(개별기업 감춤)
        self.mode_spec = mode_spec
        self.mode_spec_color = mode_spec_color
        self.mode_standard = mode_standard
        self.standard_list = standard_list
        self.industry_list = industry_list
        self.start_month = start_month # 시작연월
        self.industry_color = industry_color
        self.industry_width = industry_width
        self.indiv_color = indiv_color
        self.indiv_width = indiv_width
        self.standard_color = standard_color
        self.standard_width = standard_width

        self.ylim_auto = ylim_auto # 'yes'(최대최소에 맞춤), 'no'(0~20%, 전업권시에 권장)

        self.numlabels_on = numlabels_on
        self.fontsize = fontsize
        self.numlabels_pos = numlabels_pos
        self.xlabels_pos = xlabels_pos
        self._draw()

    def _draw(self):
        # 전처리
        major_filter = self.df.groupby(['사명', '업권', '상품구분', '연월']) \
                                .agg({'평균금리': ['mean']})['평균금리']
        major_filter_reset_index = major_filter.reset_index()

        # major_filter_industry = major_filter.loc[major_filter.index.get_level_values('업권').isin(self.industry_list)]

        standard = major_filter_reset_index[major_filter_reset_index['사명'].isin(self.standard_list)]

        industry = major_filter.reset_index()
        industry = industry.groupby(['업권','상품구분','연월']) \
                            .agg({'mean': ['mean']})['mean']
        industry = industry.reset_index()

        month_null_df = major_filter_reset_index.loc[:,['연월','mean']]
        month_null_df['mean'] = np.nan
        month_null_df = month_null_df.drop_duplicates(keep='first')


        if self.mode_standard == 'on':
            line_standard = {}
            for i in self.standard_list: # (업권, 상품구분)
                for j in list(standard[standard['사명']==i]['상품구분'].unique()):
                    tmp = standard[(standard['사명']==i) & (standard['상품구분']==j)].loc[:,['연월','mean']]

                    month_null_df_filtered = month_null_df[~month_null_df['연월'].isin(tmp['연월'])]
                    tmp = pd.concat([tmp, month_null_df_filtered], ignore_index=True)
                    tmp = tmp.sort_values(by='연월').reset_index(drop=True)
                    previous_value = 0
                    tmp['x_pos'] = [previous_value := (previous_value + (0.2 if month <= self.start_month else 1)) for month in tmp['연월']]

                    line_standard[frozenset({i, j})] = tmp

        line_industry = {}
        for i in self.industry_list: # (업권, 상품구분)
            for j in list(industry[industry['업권']==i]['상품구분'].unique()):
                tmp = industry[(industry['업권']==i) & (industry['상품구분']==j)].loc[:,['연월','mean']]
                month_null_df_filtered = month_null_df[~month_null_df['연월'].isin(tmp['연월'])]
                tmp = pd.concat([tmp, month_null_df_filtered], ignore_index=True)
                tmp = tmp.sort_values(by='연월').reset_index(drop=True)
                previous_value = 0
                tmp['x_pos'] = [previous_value := (previous_value + (0.2 if month <= self.start_month else 1)) for month in tmp['연월']]
                line_industry[frozenset({i, j})] = tmp

        if self.mode == 'on':
            line_indiv = {}
            for i in self.industry_list: # (업권, 사명, 상품구분)
                for j in list(major_filter_reset_index[major_filter_reset_index['업권']==i]['사명'].unique()):
                    for k in list(major_filter_reset_index[(major_filter_reset_index['업권']==i) & (major_filter_reset_index['사명']==j)]['상품구분'].unique()):
                        tmp = major_filter_reset_index[(major_filter_reset_index['사명']==j) & (major_filter_reset_index['상품구분']==k)].loc[:,['연월','mean']]
                        month_null_df_filtered = month_null_df[~month_null_df['연월'].isin(tmp['연월'])]
                        tmp = pd.concat([tmp, month_null_df_filtered], ignore_index=True)
                        tmp = tmp.sort_values(by='연월').reset_index(drop=True)
                        previous_value = 0
                        tmp['x_pos'] = [previous_value := (previous_value + (0.2 if month <= self.start_month else 1)) for month in tmp['연월']]

                        line_indiv[frozenset({i, j, k})] = tmp


        if self.ylim_auto == 'yes':
            y_max = major_filter_reset_index[(major_filter_reset_index['업권'].isin(self.industry_list)) & (major_filter_reset_index['연월']>=self.start_month)]['mean'].max() + 0.005
            y_min = major_filter_reset_index[(major_filter_reset_index['업권'].isin(self.industry_list)) & (major_filter_reset_index['연월']>=self.start_month)]['mean'].min() - 0.005
            self.ax.set_ylim(y_min, y_max)

        
        previous_value = 0
        line_x = [previous_value := (previous_value + (0.2 if month <= self.start_month else 1)) for month in month_null_df['연월']]
        line_x.append(line_x[-1]+1)
        xtick_midpoints = [(line_x[i] + line_x[i+1])/2 for i in range(len(line_x)-1)]
        xticklabels = [convert_month_format(month) for month in list(month_null_df['연월'])]
        for (x_pos, label) in zip(xtick_midpoints, xticklabels):
            self.ax.text(x_pos, self.ax.get_ylim()[0]-self.xlabels_pos*(self.ax.get_ylim()[1] - self.ax.get_ylim()[0]), label, ha='center', va='center', weight='ultralight', fontsize=self.fontsize)





        if self.mode == 'on':
            for i in self.industry_list:
                for j in list(major_filter_reset_index[major_filter_reset_index['업권']==i]['사명'].unique()):
                    for k in list(major_filter_reset_index[(major_filter_reset_index['업권']==i) & (major_filter_reset_index['사명']==j)]['상품구분'].unique()):
                        x_pos = list(line_indiv[frozenset({i, j, k})]['x_pos'])
                        y_pos = list(line_indiv[frozenset({i, j, k})]['mean'])
                        months = list(line_indiv[frozenset({i, j, k})]['연월'])
                        start_idx = months.index(self.start_month)

                        if self.mode_spec_color == 'on':
                            colors = plt.get_cmap('Set2')
                            colors_pos = list(major_filter_reset_index[major_filter_reset_index['업권']==i]['사명'].unique()).index(j)
                            for l in range(len(line_indiv[frozenset({i, j, k})]['연월'])-1):
                                self.ax.plot([x_pos[l], x_pos[l+1]], [y_pos[l], y_pos[l]], color=colors(colors_pos), linewidth=self.industry_width/2)
                                self.ax.plot([x_pos[l+1], x_pos[l+1]], [y_pos[l], y_pos[l+1]], color=colors(colors_pos), linewidth=self.industry_width/2)
                            self.ax.plot([x_pos[-1], x_pos[-1]+1], [y_pos[-1], y_pos[-1]], color=colors(colors_pos), linewidth=self.industry_width/2)
                        else:
                            for l in range(len(line_indiv[frozenset({i, j, k})]['연월'])-1):
                                self.ax.plot([x_pos[l], x_pos[l+1]], [y_pos[l], y_pos[l]], color=self.indiv_color[i], linewidth=self.indiv_width)
                                self.ax.plot([x_pos[l+1], x_pos[l+1]], [y_pos[l], y_pos[l+1]], color=self.indiv_color[i], linewidth=self.indiv_width)
                            self.ax.plot([x_pos[-1], x_pos[-1]+1], [y_pos[-1], y_pos[-1]], color=self.indiv_color[i], linewidth=self.indiv_width)

                        self.ax.text(x_pos[-1]+1.1, y_pos[-1], f'{j}', va='center', fontsize=self.fontsize)

                        if self.mode_spec == 'on':
                            xtick_midpoints = [(x_pos[m] + x_pos[m+1]) / 2 for m in range(start_idx, len(x_pos) - 1)]
                            xtick_midpoints.append(xtick_midpoints[-1]+1)
                            if self.numlabels_on:
                                for idx, (x_text_pos, label) in enumerate(zip(xtick_midpoints, line_indiv[frozenset({i, j, k})]['mean'])):
                                    self.ax.text(x_text_pos, list(line_indiv[frozenset({i, j, k})]['mean'])[idx+start_idx]+self.numlabels_pos*(self.ax.get_ylim()[1] - self.ax.get_ylim()[0]), format_as_percent_point1(label)
                                                , ha='center', va='center', weight='ultralight', fontsize=self.fontsize)
        
        
        for i in self.industry_list:
            for j in list(industry[industry['업권']==i]['상품구분'].unique()):
                x_pos = list(line_industry[frozenset({i, j})]['x_pos'])
                y_pos = list(line_industry[frozenset({i, j})]['mean'])
                months = list(line_industry[frozenset({i, j})]['연월'])
                start_idx = months.index(self.start_month)
                for k in range(len(line_industry[frozenset({i, j})]['연월'])-1):
                    self.ax.plot([x_pos[k], x_pos[k+1]], [y_pos[k], y_pos[k]], color=self.indiv_color[i], linewidth=self.industry_width)
                    self.ax.plot([x_pos[k+1], x_pos[k+1]], [y_pos[k], y_pos[k+1]], color=self.indiv_color[i], linewidth=self.industry_width)
                self.ax.plot([x_pos[-1], x_pos[-1]+1], [y_pos[-1], y_pos[-1]], color=self.indiv_color[i], linewidth=self.industry_width)

                if len(self.industry_list) > 1:
                    self.ax.text(x_pos[-1]+1.1, y_pos[-1], f'{i}\n({j})' if j != '일반신용' else f'{i}', va='center', fontsize=self.fontsize)
                else:
                    pass

                xtick_midpoints = [(x_pos[m] + x_pos[m+1]) / 2 for m in range(start_idx, len(x_pos) - 1)]
                xtick_midpoints.append(xtick_midpoints[-1]+1)
                labels_list = list(line_industry[frozenset({i, j})]['mean'])[start_idx:]
                for idx, (x_text_pos, label) in enumerate(zip(xtick_midpoints, labels_list)):
                    self.ax.text(x_text_pos, list(line_industry[frozenset({i, j})]['mean'])[idx+start_idx]+self.numlabels_pos*(self.ax.get_ylim()[1] - self.ax.get_ylim()[0]), format_as_percent_point1(label)
                                , ha='center', va='center', weight='ultralight', fontsize=self.fontsize)
        
                    
                    
        if self.mode_standard == 'on':
            for i in self.standard_list:
                for j in list(standard['상품구분'].unique()):
                    x_pos = list(line_standard[frozenset({i, j})]['x_pos'])
                    y_pos = list(line_standard[frozenset({i, j})]['mean'])
                    months = list(line_standard[frozenset({i, j})]['연월'])
                    start_idx = months.index(self.start_month)

                    for k in range(len(line_standard[frozenset({i, j})]['연월'])-1):
                        self.ax.plot([x_pos[k], x_pos[k+1]], [y_pos[k], y_pos[k]], color=self.standard_color, linewidth=self.standard_width)
                        self.ax.plot([x_pos[k+1], x_pos[k+1]], [y_pos[k], y_pos[k+1]], color=self.standard_color, linewidth=self.standard_width)
                        self.ax.plot([x_pos[-1], x_pos[-1]+1], [y_pos[-1], y_pos[-1]], color=self.standard_color, linewidth=self.standard_width)

                    self.ax.text(x_pos[-1]+1.1, y_pos[-1], f'{i}_{j}' if j != '일반신용' else f'{i}', va='center', fontsize=self.fontsize)

                    xtick_midpoints = [(x_pos[m] + x_pos[m+1]) / 2 for m in range(start_idx, len(x_pos) - 1)]
                    xtick_midpoints.append(xtick_midpoints[-1]+1)
                    labels_list = list(line_standard[frozenset({i, j})]['mean'])[start_idx:]
                    for idx, (x_text_pos, label) in enumerate(zip(xtick_midpoints, labels_list)):
                        self.ax.text(x_text_pos, list(line_standard[frozenset({i, j})]['mean'])[idx+start_idx]+self.numlabels_pos*(self.ax.get_ylim()[1] - self.ax.get_ylim()[0]), format_as_percent_point1(label)
                                    , ha='center', va='center', weight='ultralight', fontsize=self.fontsize)



        self.ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))

        self.ax.grid(True, axis='y', which='both', linestyle='--', color='gray', linewidth=0.2, alpha=0.5)

        self.ax.set_xticklabels([])

        self.ax.set_xticks(line_x)
        self.ax.tick_params(axis='x', which='major', length=10, width=2)


        self.ax.spines[['top','right']].set_visible(False)
        self.ax.spines[['left','bottom']].set_linewidth(0.3)  # y축 spine을 얇게 설정

        self.ax.tick_params(axis='both', which='both', width=0.3)

        self.ax.tick_params(axis='y', labelsize=self.fontsize)  # 폰트 크기 설정

In [ ]:
fig = plt.figure(figsize=(11.69, 8.27))

# GridSpec으로 레이아웃 생성
gs = fig.add_gridspec(4, 4, wspace=0.4)

# 첫 번째 큰 그래프
ax_tot = fig.add_subplot(gs[:, 0:2])

# 두 번째 줄의 3개 그래프에서 y축을 공유
ax_1_1 = fig.add_subplot(gs[0, 2]) # 인터넷은행
ax_1_2 = fig.add_subplot(gs[1, 2]) # 지방은행
ax_2 = fig.add_subplot(gs[0:2, 3]) # 신용카드
ax_3 = fig.add_subplot(gs[2:4, 2]) # 저축은행
ax_4 = fig.add_subplot(gs[2:4, 3]) # 캐피탈

# 전체 배경색 설정
# fig.patch.set_facecolor("#ddddff")

# 전업권 평균
tot_step = int_step_graph(
    ax_tot
    , df = tot_cs_major[~(tot_cs_major['상품구분']=='한도')]
    , mode = 'off'
    , mode_spec = 'off'
    , mode_spec_color = 'off'
    , mode_standard= 'on'
    , industry_list = ['시중은행','인터넷은행','지방은행','신용카드','캐피탈','저축은행']
    , standard_list = ['OK']
    , start_month ='202401'
    , industry_color='black'
    , industry_width=2
    , indiv_color = {'시중은행':'black', '인터넷은행':'dimgray', '지방은행':'darkgray', '신용카드':'blue', '캐피탈':'green', '저축은행':'red'}
    , indiv_width=0.2
    , standard_color ='darkorange'
    , standard_width=2
    , ylim_auto='yes'
    , fontsize=7
    , numlabels_pos=0.005
    , xlabels_pos=0.02
    , numlabels_on=True
)

# 인터넷은행
online_step = int_step_graph(
    ax_1_1
    , df = tot_cs_major[~(tot_cs_major['상품구분']=='한도')]
    , mode = 'on'
    , mode_spec = 'off'
    , mode_spec_color = 'on'
    , mode_standard= 'off'
    , industry_list = ['인터넷은행']
    , standard_list = ['OK']
    , start_month ='202401'
    , industry_color='black'
    , industry_width=2
    , indiv_color = {'시중은행':'black', '인터넷은행':'dimgray', '지방은행':'darkgray', '신용카드':'blue', '캐피탈':'green', '저축은행':'red'}
    , indiv_width=0.2
    , standard_color ='darkorange'
    , standard_width=2
    , ylim_auto='yes'
    , fontsize=4
    , numlabels_pos=0.005
    , xlabels_pos=0.05
    , numlabels_on=True
)

# 지방은행
online_step = int_step_graph(
    ax_1_2
    , df = tot_cs_major[~(tot_cs_major['상품구분']=='한도')]
    , mode = 'on'
    , mode_spec = 'off'
    , mode_spec_color = 'on'
    , mode_standard= 'off'
    , industry_list = ['지방은행']
    , standard_list = ['OK']
    , start_month ='202401'
    , industry_color='black'
    , industry_width=2
    , indiv_color = {'시중은행':'black', '인터넷은행':'dimgray', '지방은행':'darkgray', '신용카드':'blue', '캐피탈':'green', '저축은행':'red'}
    , indiv_width=0.2
    , standard_color ='darkorange'
    , standard_width=2
    , ylim_auto='yes'
    , fontsize=4
    , numlabels_pos=0.005
    , xlabels_pos=0.05
    , numlabels_on=True
)

# 신용카드
online_step = int_step_graph(
    ax_2
    , df = tot_cs_major[~(tot_cs_major['상품구분']=='한도')]
    , mode = 'on'
    , mode_spec = 'off'
    , mode_spec_color = 'on'
    , mode_standard= 'off'
    , industry_list = ['신용카드']
    , standard_list = ['OK']
    , start_month ='202401'
    , industry_color='black'
    , industry_width=2
    , indiv_color = {'시중은행':'black', '인터넷은행':'dimgray', '지방은행':'darkgray', '신용카드':'blue', '캐피탈':'green', '저축은행':'red'}
    , indiv_width=0.2
    , standard_color ='darkorange'
    , standard_width=2
    , ylim_auto='yes'
    , fontsize=4
    , numlabels_pos=0.005
    , xlabels_pos=0.02
    , numlabels_on=True
)

# 저축은행
online_step = int_step_graph(
    ax_3
    , df = tot_cs_major[~(tot_cs_major['상품구분']=='한도')]
    , mode = 'on'
    , mode_spec = 'off'
    , mode_spec_color = 'on'
    , mode_standard= 'off'
    , industry_list = ['저축은행']
    , standard_list = ['OK']
    , start_month ='202401'
    , industry_color='black'
    , industry_width=2
    , indiv_color = {'시중은행':'black', '인터넷은행':'dimgray', '지방은행':'darkgray', '신용카드':'blue', '캐피탈':'green', '저축은행':'red'}
    , indiv_width=0.2
    , standard_color ='darkorange'
    , standard_width=2
    , ylim_auto='yes'
    , fontsize=4
    , numlabels_pos=0.005
    , xlabels_pos=0.02
    , numlabels_on=True
)

# 캐피탈
online_step = int_step_graph(
    ax_4
    , df = tot_cs_major[~(tot_cs_major['상품구분']=='한도')]
    , mode = 'on'
    , mode_spec = 'off'
    , mode_spec_color = 'on'
    , mode_standard= 'off'
    , industry_list = ['캐피탈']
    , standard_list = ['OK']
    , start_month ='202401'
    , industry_color='black'
    , industry_width=2
    , indiv_color = {'시중은행':'black', '인터넷은행':'dimgray', '지방은행':'darkgray', '신용카드':'blue', '캐피탈':'green', '저축은행':'red'}
    , indiv_width=0.2
    , standard_color ='darkorange'
    , standard_width=2
    , ylim_auto='yes'
    , fontsize=4
    , numlabels_pos=0.005
    , xlabels_pos=0.02
    , numlabels_on=True
)


plt.savefig('test_avg_step.png', bbox_inches='tight', dpi=600)
plt.show()

#### 4.1.2. 정리 테이블

In [ ]:
avg_int_df = tot_cs_major.drop(columns=['신용점수구간','구간평균금리'], axis=1)
avg_int_df = avg_int_df.drop_duplicates()
avg_int_df

avg_int_pivot = avg_int_df[~(avg_int_df['상품구분']=='한도')].pivot_table(index=['업권','상품구분','사명'], columns='연월', values='평균금리')
avg_int_pivot = pd.merge(avg_int_pivot, tot_bal_major[tot_bal_major['연월'].isin([202403, '2024Q1'])][['사명','잔액']], on='사명', how='left')
avg_int_pivot
avg_int_pivot.to_csv('평균금리 정리.csv', encoding='euc-kr')

### 4.2. 신용점수 구간대별 평균금리와 금리대별 신규취급비중

#### 4.2.1. 그래프

In [ ]:
''' 함수상세

타겟 테이블 - tot_cs_major

'''

class cs_viline_graph:
    def __init__(self, ax, df, standard_nm, cs_dic, industry_list):
        self.ax = ax

        self.df = df
        self.standard_nm = standard_nm
        self.cs_dic = cs_dic
        self.industry_list = industry_list

        self._draw()

    def _draw(self):
        # 전처리
        major_filter = self.df[self.df['연월']==self.df['연월'].max()] \
                                .groupby(['사명', '업권', '상품구분', '평균금리', '신용점수구간'], sort=False) \
                                .agg({'구간평균금리': ['mean']})['구간평균금리']

        # 원래 순서 유지
        major_filter = major_filter.groupby(['사명', '업권', '상품구분', '평균금리'], group_keys=False, sort=False).apply(process_group_cummax)
        major_filter.dropna(axis=0, how='any', inplace=True)
        major_filter.drop(columns='cummax_mean', inplace=True)

        major_filter_reset_index = major_filter.reset_index()

        # 업권 리스트 필터링 후 처리
        major_filter_industry = major_filter.loc[major_filter.index.get_level_values('업권').isin(self.industry_list)]

        # 기준 설정
        standard = major_filter_reset_index[major_filter_reset_index['사명']==self.standard_nm]

        # 업권별 처리
        industry = major_filter.reset_index()
        industry = industry[industry['업권'].isin(self.industry_list)]
        industry = industry.groupby(['업권','상품구분','신용점수구간'], sort=False) \
                                    .agg({'mean': ['mean','count']})['mean'] \
                                    .query("count > (count.max() / 3)") \
                                    .groupby(['업권', '상품구분'], sort=False)['mean'].cummax() \
                                    .drop_duplicates(keep='first')
        industry = industry.reset_index()

        industry_avg = major_filter.reset_index()
        industry_avg = industry_avg[industry_avg['업권'].isin(industry_list)]
        industry_avg = industry_avg.groupby(['업권','상품구분'], sort=False) \
                                    .agg({'평균금리': ['mean']})['평균금리']
        industry_avg = industry_avg.reset_index()

        # 프레임설정
        vlines_cnt = 1 + len(industry['업권'].unique()) + len(major_filter_industry.reset_index(level='사명')['사명'].unique())

        vline_standard_x_pos = [0, 1]

        industry_len = len(industry.drop_duplicates(subset=['업권', '상품구분'])['업권']) * 2
        vline_industry_x_pos = [1.2]
        for i in range(1, industry_len):
            increment = 1 if i % 2 != 0 else 0.2
            vline_industry_x_pos.append(vline_industry_x_pos[-1] + increment)

        major_len = len(major_filter_industry.reset_index().drop_duplicates(subset=['사명', '상품구분'])['사명']) * 2
        vline_major_x_pos = [vline_industry_x_pos[-1]+0.2]
        for i in range(1, major_len):
            increment = 0.5 if i % 2 != 0 else 0.2
            vline_major_x_pos.append(vline_major_x_pos[-1] + increment)

        self.ax.vlines(x= vline_standard_x_pos, ymin=0, ymax=0.2, color='orange', linewidth=4, alpha=0)
        self.ax.vlines(x= vline_industry_x_pos, ymin=0, ymax=0.2, color='black', linewidth=2, alpha=0)
        self.ax.vlines(x= vline_major_x_pos, ymin=0, ymax=0.2, color='gray', linewidth=1, alpha=0)
        
        # 위아래 금리표현
        y_hlines = [0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.14, 0.16, 0.18]
        y_hlines_label = ['2%','4%','6%','8%','10%','12%','14%','16%','18%']
        self.ax.hlines(y=y_hlines, xmin=0, xmax=vline_major_x_pos[-1], color='black', linewidth=0.25, linestyle='--', alpha=0.25)
        for y, label in zip(y_hlines, y_hlines_label):
            self.ax.text(vline_major_x_pos[-1] + 0.1, y, label, va='center', rotation=90, fontsize=8)
            self.ax.text(-0.5, y, label, va='center', rotation=90, fontsize=8)

            
        for i in range(len(vline_standard_x_pos) // 2):
            if 2 * i  < len(vline_standard_x_pos):
                self.ax.axvspan(vline_standard_x_pos[2 * i], vline_standard_x_pos[2 * i + 1], color='gray', alpha=0.6)

        for i in range(len(vline_industry_x_pos) // 2):
            if 2 * i + 1 < len(vline_industry_x_pos):
                self.ax.axvspan(vline_industry_x_pos[2 * i], vline_industry_x_pos[2 * i + 1], color='gray', alpha=0.4)

        for i in range(len(vline_major_x_pos) // 2):
            if 2 * i + 1 < len(vline_major_x_pos):
                self.ax.axvspan(vline_major_x_pos[2 * i], vline_major_x_pos[2 * i + 1], color='gray', alpha=0.2)

        unique_values = sorted(major_filter_reset_index['신용점수구간'].unique().tolist(), key=int)
        second_digits = [value[1] for value in unique_values]
        unique_second_digits = sorted(set(second_digits))
        cmap = plt.cm.get_cmap('YlOrRd', len(unique_second_digits))
        norm = plt.Normalize(min(unique_second_digits), max(unique_second_digits))
        color_dict = {digit: cmap(norm(int(digit))) for digit in unique_second_digits}
        final_color_dict = {value: color_dict[value[1]] for value in unique_values}

        # 기준 기업의 신용점수 구간별 평균금리 수직 오더링
        for y, label in zip(standard['mean'].tolist(), standard['신용점수구간'].unique().tolist()):
            plt.hlines(y, xmin=0, xmax=1, color=final_color_dict[label], linewidth=4)
            plt.text(0.9, y, cs_code_easy[label], va='center', ha='center', rotation=90, fontsize=6)
        plt.hlines(standard['평균금리'].mean(), xmin=0, xmax=1, color='dimgray', linewidth=2)
        plt.text(1.4, standard['평균금리'].mean(), f"{standard['평균금리'].mean()*100:.1f}%", fontsize=7, va='center', ha='center', rotation=90)
        
        
        # 업권 평균 신용점수 구간별 평균금리 수직 오더링
        industry_prod_tuple = industry.drop_duplicates(subset=['업권', '상품구분'])
        industry_prod_tuple = [tuple(x) for x in industry_prod_tuple[['업권', '상품구분']].values]
        for idx, indiv_prod in enumerate(industry_prod_tuple):
            for y, label in zip(industry[(industry['업권']==indiv_prod[0]) & (industry['상품구분']==indiv_prod[1])]['mean'].tolist(), industry[(industry['업권']==indiv_prod[0]) & (industry['상품구분']==indiv_prod[1])]['신용점수구간'].unique().tolist()):
                plt.hlines(y, xmin=vline_industry_x_pos[idx*2], xmax=vline_industry_x_pos[idx*2+1], color=final_color_dict[label], linewidth=5)
                plt.text((vline_industry_x_pos[idx*2]+vline_industry_x_pos[idx*2+1])/2+0.4, y, cs_code_easy[label], va='center', ha='center', rotation=90, fontsize=6)
            plt.hlines(industry_avg[(industry_avg['업권']==indiv_prod[0]) & (industry_avg['상품구분']==indiv_prod[1])]['mean'], xmin=vline_industry_x_pos[idx*2], xmax=vline_industry_x_pos[idx*2+1], color='dimgray', linewidth=2)
            plt.text(vline_industry_x_pos[idx*2+1] + 0.4, industry_avg[(industry_avg['업권']==indiv_prod[0]) & (industry_avg['상품구분']==indiv_prod[1])]['mean'].mean(), f"{industry_avg[(industry_avg['업권']==indiv_prod[0]) & (industry_avg['상품구분']==indiv_prod[1])]['mean'].mean()*100:.1f}%", fontsize=7, va='center', ha='center', rotation=90)

            
        # 주요 기업 신용점수 구간별 평균금리 수직 오더링
        major_filter_industry = major_filter_industry.reset_index()
        major_prod_tuple = major_filter_industry.drop_duplicates(subset=['사명', '상품구분'])
        major_prod_tuple = [tuple(x) for x in major_prod_tuple[['사명', '상품구분']].values]
        for idx, indiv_prod in enumerate(major_prod_tuple):
            for y, label in zip(major_filter_reset_index[(major_filter_reset_index['사명']==indiv_prod[0]) & (major_filter_reset_index['상품구분']==indiv_prod[1])]['mean'].tolist(), major_filter_reset_index[(major_filter_reset_index['사명']==indiv_prod[0]) & (major_filter_reset_index['상품구분']==indiv_prod[1])]['신용점수구간'].unique()):
                plt.hlines(y, xmin=vline_major_x_pos[idx*2], xmax=vline_major_x_pos[idx*2+1], color=final_color_dict[label], linewidth=4, label=label)
            plt.hlines(major_filter_reset_index[(major_filter_reset_index['사명']==indiv_prod[0]) & (major_filter_reset_index['상품구분']==indiv_prod[1])]['평균금리'].mean(), xmin=vline_major_x_pos[idx*2], xmax=vline_major_x_pos[idx*2+1], color='darkgray', linewidth=2)


        self.ax.spines['right'].set_visible(False)
        self.ax.spines['left'].set_visible(False)
        self.ax.spines['bottom'].set_visible(False)
        self.ax.spines['top'].set_visible(False)

        self.ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))

        self.ax.set_xlim(0,vline_major_x_pos[-1])
        self.ax.set_ylim(0,0.2)

        # self.ax.yaxis.set_inverted(True)

        current_xticks = self.ax.get_xticks()
        current_yticks = self.ax.get_yticks()

        # self.ax.tick_params(top=True, labeltop=True, axis='x')

        self.ax.set_yticks([])          # X축의 틱 제거
        self.ax.set_yticklabels([])

        # self.ax.legend()

        # 원하는 위치 지정
        xtick_labels_pos_tmp = vline_standard_x_pos + vline_industry_x_pos + vline_major_x_pos
        xtick_labels_pos = [(xtick_labels_pos_tmp[i] + xtick_labels_pos_tmp[i + 1]) / 2 for i in range(0, len(xtick_labels_pos_tmp), 2) if i + 1 < len(xtick_labels_pos_tmp)]

        
        # 라벨: 회사이름
        xtick_labels = ([self.standard_nm] +
                        [f'{t[0]}_{t[1]}' if t[1] != '일반신용' else f'{t[0]}' for t in industry_prod_tuple] +
                        [f'{t[0]}_{t[1]}' if t[1] != '일반신용' else f'{t[0]}' for t in major_prod_tuple]
                    )

        # 위치와 라벨을 설정
        self.ax.set_xticks(xtick_labels_pos)
        self.ax.set_xticklabels(xtick_labels, rotation=90, fontsize=8)

        self.ax.tick_params(axis='y', labelrotation=90)



        # 컬러바 테스트
        # 범례 추가를 위한 컬러맵 설정
        cmap = plt.cm.get_cmap('YlOrRd', len(unique_second_digits))
        norm = plt.Normalize(vmin=min(unique_second_digits), vmax=max(unique_second_digits))

        # 컬러바 생성
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])

        # Colorbar 추가
        cbar = plt.colorbar(sm, ax=self.ax, shrink=0.6, aspect=50)

        cbar.outline.set_visible(False)

        # unique_second_digits를 숫자로 변환하여 틱 설정
        unique_second_digits_numeric = [int(digit) for digit in unique_second_digits]

        # 틱 위치를 색상 구간의 중앙으로 설정
        n_colors = len(unique_second_digits_numeric)
        tick_positions = np.linspace(min(unique_second_digits_numeric) + 0.5, max(unique_second_digits_numeric) - 0.5, n_colors)

        # 틱 설정 및 레이블 추가
        cbar.set_ticks(tick_positions)
        cbar.set_ticklabels([cs_code_easy_rev[digit] for digit in unique_second_digits], fontsize=6)  # 구간 레이블 표시

        for label in cbar.ax.get_yticklabels():
            label.set_rotation(90)  # 90도 회전
            label.set_ha('center')  # 가로 정렬을 가운데로 설정
            label.set_va('center') 

        cbar.ax.tick_params(size=0)

In [ ]:
fig, ax = plt.subplots(figsize=(11.7, 8))
industry_list = ['시중은행','인터넷은행','지방은행','신용카드','저축은행','캐피탈']
test = cs_viline_graph(ax, tot_cs_major[~(tot_cs_major['상품구분']=='한도')], 'OK', cs_code_rev, industry_list)


plt.savefig('test_cs_vline.png', bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
cs_pivot = tot_cs_major[~(tot_cs_major['상품구분']=='한도') & (tot_cs_major['연월']=='202409')].pivot_table(index=['업권','상품구분','사명'], columns='신용점수구간', values='구간평균금리')
cs_pivot = pd.merge(cs_pivot, tot_bal_major[tot_bal_major['연월'].isin([202403, '2024Q1'])][['사명','잔액']], on='사명', how='left')
cs_pivot
cs_pivot.to_csv('신용점수구간별 평균금리.csv', encoding='euc-kr')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


# 데이터 생성
y = [4.8, 5.5, 3.5, 4.6, 6.5, 6.6, 2.6, 3.0]

# 그래프 그리기
fig, ax = plt.subplots()

# 가로 방향 계단 그래프 생성, 양 끝 수직선 없애기 (baseline=None)
ax.stairs(y, orientation='horizontal', linewidth=2.5)

# 축 설정
ax.set(ylim=(0, 9), yticks=np.arange(1, 8),
       xlim=(0, 9), xticks=np.arange(1, 8))

plt.show()

### 4.3. 금리구간별 취급비중

In [ ]:
int_pivot = tot_int_major[~(tot_int_major['상품구분']=='한도')&(tot_int_major['연월']=='202409')].pivot_table(index=['업권','상품구분','사명'], columns='금리구간', values='취급비중')
int_pivot = pd.merge(int_pivot, tot_bal_major[tot_bal_major['연월'].isin([202403, '2024Q1'])][['사명','잔액']], on='사명', how='left')

int_pivot.to_csv('금리구간별 취급비중.csv', encoding='euc-kr')

In [ ]:
bank_cs_standard.groupby(['사명','업권','상품구분','연월']).mean()
card_cs_loan
card_cs_cash
savings_cs
capital_cs

In [ ]:
len(bank_cs_standard['연월'].unique())
len(card_cs_loan['연월'].unique())
len(card_cs_cash['연월'].unique())
len(savings_cs['연월'].unique())
len(capital_cs['연월'].unique())

.to_csv('은행업권 평균금리.csv', encoding='euc-kr')

In [ ]:
plt.stairs(bank_cs_standard['연월'], range(len(bank_cs_standard['연월'].unique()))

In [ ]:
print(major_nationwide)
print(major_local)
print(major_internet)
print(major_savings)
print(major_card)
print(major_capital)

In [ ]:
## 주요개별사
bank_cs_standard.groupby(['사명','업권','상품구분','연월'])[['평균금리']].mean()

## 업권(주요개별사)
bank_grouping = bank_cs_standard.groupby(['업권','상품구분','연월'])[['평균금리']].mean()

In [ ]:
industry_mean

In [ ]:
indiv_bank_nationwide_mean = bank_cs_standard[bank_cs_standard['사명'].isin(major_nationwide)].groupby(['사명','업권','상품구분','연월'])[['평균금리']].mean()
indiv_bank_local_mean = bank_cs_standard[bank_cs_standard['사명'].isin(major_local)].groupby(['사명','업권','상품구분','연월'])[['평균금리']].mean()
indiv_bank_internet_mean = bank_cs_standard[bank_cs_standard['사명'].isin(major_internet)].groupby(['사명','업권','상품구분','연월'])[['평균금리']].mean()
indiv_savings_mean = savings_cs[savings_cs['사명'].isin(major_savings)].groupby(['사명','업권','상품구분','연월'])[['평균금리']].mean()
indiv_card_loan_mean = card_cs_loan[card_cs_loan['사명'].isin(major_card)].groupby(['사명','업권','상품구분','연월'])[['평균금리']].mean()
indiv_card_cash_mean = card_cs_cash[card_cs_cash['사명'].isin(major_card)].groupby(['사명','업권','상품구분','연월'])[['평균금리']].mean()
indiv_capital_mean = capital_cs[capital_cs['사명'].isin(major_capital)].groupby(['사명','업권','상품구분','연월'])[['평균금리']].mean()

bank_nationwide_mean = bank_cs_standard[bank_cs_standard['사명'].isin(major_nationwide)].groupby(['업권','상품구분','연월'])[['평균금리']].mean()
bank_local_mean = bank_cs_standard[bank_cs_standard['사명'].isin(major_local)].groupby(['업권','상품구분','연월'])[['평균금리']].mean()
bank_internet_mean = bank_cs_standard[bank_cs_standard['사명'].isin(major_internet)].groupby(['업권','상품구분','연월'])[['평균금리']].mean()
savings_mean = savings_cs[savings_cs['사명'].isin(major_savings)].groupby(['업권','상품구분','연월'])[['평균금리']].mean()
card_loan_mean = card_cs_loan[card_cs_loan['사명'].isin(major_card)].groupby(['업권','상품구분','연월'])[['평균금리']].mean()
card_cash_mean = card_cs_cash[card_cs_cash['사명'].isin(major_card)].groupby(['업권','상품구분','연월'])[['평균금리']].mean()
capital_mean = capital_cs[capital_cs['사명'].isin(major_capital)].groupby(['업권','상품구분','연월'])[['평균금리']].mean()

industry_mean = pd.concat([bank_nationwide_mean,bank_local_mean,bank_internet_mean,savings_mean,card_loan_mean,card_cash_mean,capital_mean], ignore_index =False)
industry_mean = industry_mean.reset_index()
industry_mean['사명'] = industry_mean['업권']
industry_mean = industry_mean[['사명','업권','상품구분','연월','평균금리']]
industry_mean['연월'] = industry_mean['연월'].astype(str)
industry_mean['연분기'] = (
    industry_mean['연월'].str[:4] + 'Q' + ((industry_mean['연월'].str[4:].astype(int) -1) // 3 + 1).astype(str)
)
indiv_mean = pd.concat([indiv_bank_nationwide_mean,indiv_bank_local_mean,indiv_bank_internet_mean,indiv_savings_mean,indiv_card_loan_mean,indiv_card_cash_mean,indiv_capital_mean], ignore_index =False)
indiv_mean = indiv_mean.reset_index()
indiv_mean['연월'] = indiv_mean['연월'].astype(str)
indiv_mean['연분기'] = (
    indiv_mean['연월'].str[:4] + 'Q' + ((indiv_mean['연월'].str[4:].astype(int) -1) // 3 + 1).astype(str)
)
int_mean = pd.concat([industry_mean, indiv_mean], ignore_index=False)
int_mean.to_csv('신용대출금리.csv', encoding='euc-kr', index=False)


In [ ]:
savings_cs.to_csv('저축은행 신용점수별.csv', encoding='euc-kr', index=False)

In [ ]:
savings_int.to_csv('저축은행 금리대별.csv', encoding='euc-kr', index=False)